# 6. Preflight A:
      
   architecture = model_a_nodal_multiscale_pair
   483 variantes biológicas
   + 1 WT control NO entrenable
   split = 342 / 78 / 63
   split.allow_create = false
   escalas = mutation/local/global
   domain desactivada
   run_root exclusivamente A
   fresh run

In [2]:
# ============================================================
# A9 MODELO A — PARÁMETROS OPERATIVOS CONGELADOS
# ============================================================

REPO_URL = 'https://github.com/sap15/ViVU_lab.git'

GIT_COMMIT = '3757bfdde52a9920611f3e7b8027434ce80d54bc'

RUN_SEED = 11
SPLIT_SEED = 42

ARCHITECTURE = 'model_a_nodal_multiscale_pair'

DRIVE_PROJECT_BASE = (
    '/content/drive/MyDrive/modelos_proyecto_PKP2'
)

DRIVE_MODEL_A_BASE = (
    DRIVE_PROJECT_BASE + '/model_a'
)

DRIVE_MUTANTS_HDF5 = (
    DRIVE_MODEL_A_BASE + '/data/proc_483p.hdf5'
)

DRIVE_WT_HDF5 = (
    DRIVE_MODEL_A_BASE + '/data/wt_companion.hdf5'
)

DRIVE_MODEL_A_A9_ROOT = (
    DRIVE_MODEL_A_BASE + '/runs/model_a_a9'
)

# Se necesita también un namespace B independiente para que
# el preflight valide que A/B nunca comparten outputs.
DRIVE_MODEL_B_A9_ROOT = (
    DRIVE_PROJECT_BASE + '/model_b/runs/model_b_a9'
)

LOCAL_ROOT = '/content/model_a_workspace'
REPO_DIR = LOCAL_ROOT + '/repo'
STAGING_ROOT = LOCAL_ROOT + '/staging'

print('GIT_COMMIT =', GIT_COMMIT)
print('RUN_SEED   =', RUN_SEED)
print('SPLIT_SEED =', SPLIT_SEED)
print('ARCH       =', ARCHITECTURE)

GIT_COMMIT = 3757bfdde52a9920611f3e7b8027434ce80d54bc
RUN_SEED   = 11
SPLIT_SEED = 42
ARCH       = model_a_nodal_multiscale_pair


In [3]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

for label, raw_path in [
    ('mutants', DRIVE_MUTANTS_HDF5),
    ('wt_companion', DRIVE_WT_HDF5),
]:
    path = Path(raw_path)

    print(
        label,
        'exists=', path.is_file(),
        'size=', path.stat().st_size if path.is_file() else None,
        'path=', path,
    )

    if not path.is_file():
        raise FileNotFoundError(
            f'No existe el HDF5 requerido: {path}'
        )

    if path.stat().st_size <= 0:
        raise RuntimeError(
            f'El HDF5 está vacío: {path}'
        )

Mounted at /content/drive
mutants exists= True size= 33580891 path= /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/data/proc_483p.hdf5
wt_companion exists= True size= 22961913 path= /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/data/wt_companion.hdf5


In [4]:
from pathlib import Path
import shutil
import subprocess
import sys

repo = Path(REPO_DIR).resolve()
local_root = Path(LOCAL_ROOT).resolve()

# Queremos un clone limpio de esta sesión.
if repo.exists():
    shutil.rmtree(repo)

local_root.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [
        'git',
        'clone',
        '--no-checkout',
        REPO_URL,
        str(repo),
    ],
    check=True,
)

subprocess.run(
    [
        'git',
        '-C',
        str(repo),
        'fetch',
        '--tags',
        '--prune',
        'origin',
    ],
    check=True,
)

subprocess.run(
    [
        'git',
        '-C',
        str(repo),
        'checkout',
        '--detach',
        GIT_COMMIT,
    ],
    check=True,
)

HEAD = subprocess.run(
    [
        'git',
        '-C',
        str(repo),
        'rev-parse',
        'HEAD',
    ],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

STATUS = subprocess.run(
    [
        'git',
        '-C',
        str(repo),
        'status',
        '--porcelain',
    ],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

print('HEAD =', HEAD)
print('working_tree_clean =', STATUS == '')

assert HEAD == GIT_COMMIT
assert STATUS == ''

sys.path[:0] = [
    str(repo),
    str(repo / 'src'),
]

HEAD = 3757bfdde52a9920611f3e7b8027434ce80d54bc
working_tree_clean = True


In [5]:
import os
import platform
import torch

os.chdir(repo)

print('Python =', platform.python_version())
print('Torch =', torch.__version__)
print('CUDA available =', torch.cuda.is_available())

if torch.cuda.is_available():
    print('CUDA version =', torch.version.cuda)
    print('GPU =', torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        'A9 requiere GPU CUDA y esta sesión no la reconoce.'
    )


Python = 3.13.15
Torch = 2.11.0+cu128
CUDA available = True
CUDA version = 12.8
GPU = Tesla T4


In [6]:
from scripts.colab_preflight import prepare_colab_environment

ENVIRONMENT = prepare_colab_environment(
    repo_root=repo,
    marker_root=local_root,
    commit=HEAD,
    requirements_path=repo / 'requirements-colab.txt',
    device='cuda',
    python_version=platform.python_version(),
    torch_version=torch.__version__,
)

RUNTIME = ENVIRONMENT['runtime']

print(RUNTIME)

{'python': '3.13.15', 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.39', 'torch': '2.11.0+cu128', 'torch_geometric': '2.8.0.post1', 'cuda_visible': True, 'torch_cuda': '12.8', 'gpu_name': 'Tesla T4', 'selected_device': 'cuda', 'device_tensor_test': 'passed', 'local_disk': {'total': 120942624768, 'used': 50926899200, 'free': 69998948352}}


In [7]:
from pathlib import Path

from scripts.colab_preflight import (
    stage_file,
    require_free_space,
)

staging = Path(STAGING_ROOT).resolve()
staging.mkdir(parents=True, exist_ok=True)

required_bytes = (
    Path(DRIVE_MUTANTS_HDF5).stat().st_size
    + Path(DRIVE_WT_HDF5).stat().st_size
)

require_free_space(
    staging,
    required_bytes * 2,
)

MUTANTS_RECORD = stage_file(
    DRIVE_MUTANTS_HDF5,
    staging / 'proc_483p.hdf5',
    staging_root=staging,
    role='mutants',
)

WT_RECORD = stage_file(
    DRIVE_WT_HDF5,
    staging / 'wt_companion.hdf5',
    staging_root=staging,
    role='wt_companion',
)

# Entradas locales solo lectura.
Path(MUTANTS_RECORD['local_locator']).chmod(0o444)
Path(WT_RECORD['local_locator']).chmod(0o444)

print('MUTANTS')
print(MUTANTS_RECORD)

print('\nWT COMPANION')
print(WT_RECORD)

MUTANTS
{'role': 'mutants', 'drive_locator': '/content/drive/MyDrive/modelos_proyecto_PKP2/model_a/data/proc_483p.hdf5', 'local_locator': '/content/model_a_workspace/staging/proc_483p.hdf5', 'sha256': '92eb242f5565db6194a5e29e3469775fbf7d8c07280ed2d8cdfd5ab98b5b5631', 'size_bytes': 33580891, 'reused': False}

WT COMPANION
{'role': 'wt_companion', 'drive_locator': '/content/drive/MyDrive/modelos_proyecto_PKP2/model_a/data/wt_companion.hdf5', 'local_locator': '/content/model_a_workspace/staging/wt_companion.hdf5', 'sha256': '29f68e98ae300207511594e0baf7621ba9e67c85d8df12ceee812a1dea3aa91a', 'size_bytes': 22961913, 'reused': False}


In [8]:
import json
from pathlib import Path

from scripts.a9_preflight import validate_a9_colab_preflight

A9_PREFLIGHT = validate_a9_colab_preflight(
    repo / 'configs/model_a_a9.yaml',

    architecture=ARCHITECTURE,
    run_seed=RUN_SEED,

    mutants_hdf5=MUTANTS_RECORD['local_locator'],
    wt_hdf5=WT_RECORD['local_locator'],

    output_root=DRIVE_MODEL_A_A9_ROOT,
    peer_output_root=DRIVE_MODEL_B_A9_ROOT,

    repo_root=repo,
    expected_commit=GIT_COMMIT,

    allowed_output_root=DRIVE_PROJECT_BASE,
)

print(
    json.dumps(
        A9_PREFLIGHT,
        indent=2,
        sort_keys=True,
        default=str,
    )
)

if A9_PREFLIGHT.get('status') != 'PASS':
    raise RuntimeError(
        'A9 preflight no ha terminado en PASS.'
    )

print('\nA9_PREFLIGHT=PASS')

{
  "ab_intersection": 483,
  "git": {
    "commit": "3757bfdde52a9920611f3e7b8027434ce80d54bc",
    "expected_commit": "3757bfdde52a9920611f3e7b8027434ce80d54bc",
    "working_tree": "clean"
  },
  "inventories": {
    "model_a": {
      "biological_variants": 483,
      "dataset_identity": {
        "actual_combined_fingerprint": "9a320862a54566d232e1ef2a4eafa468cf900a19186dc0fd7c92aaf0abe06c68",
        "actual_mutant_sha256": "92eb242f5565db6194a5e29e3469775fbf7d8c07280ed2d8cdfd5ab98b5b5631",
        "actual_wt_sha256": "29f68e98ae300207511594e0baf7621ba9e67c85d8df12ceee812a1dea3aa91a",
        "dataset_identity_status": "PASS",
        "expected_combined_fingerprint": "9a320862a54566d232e1ef2a4eafa468cf900a19186dc0fd7c92aaf0abe06c68",
        "expected_mutant_sha256": "92eb242f5565db6194a5e29e3469775fbf7d8c07280ed2d8cdfd5ab98b5b5631",
        "expected_wt_sha256": "29f68e98ae300207511594e0baf7621ba9e67c85d8df12ceee812a1dea3aa91a",
        "identity_source": "/content/model_a_works

# **CALCULO DE TIEMPO ESTIMADO DE ENTRENAMIENTO**

Solo hay que cambiar el parametro "RUN_SEED" segun el id de la prueba 37,27...

In [ ]:
from pathlib import Path
import json
import math
import subprocess
from datetime import datetime, timezone

# ============================================================
# MODELO A · ETA MULTISEED
# SOLO CAMBIAR ESTA LÍNEA
# ============================================================

RUN_SEED = 23       # <-- cambiar a 37, 41 o 53 cuando corresponda

# ============================================================
# NO MODIFICAR A PARTIR DE AQUÍ
# ============================================================

MIN_EPOCHS_FOR_ETA = 10
MAX_EPOCHS = 100
DEFAULT_PATIENCE = 15

A9_ROOT = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_a/runs/model_a_a9"
)

LAUNCH_RECORD = (
    A9_ROOT
    / "_launcher_logs"
    / f"model_a_seed{RUN_SEED}_launcher.json"
)

print("=" * 90)
print(f"MODELO A · SEED {RUN_SEED} · ETA CHECK")
print("=" * 90)

# ------------------------------------------------------------
# 1. Recuperar el run correcto de esa seed
# ------------------------------------------------------------

assert LAUNCH_RECORD.is_file(), (
    f"No encuentro el launcher persistente de seed {RUN_SEED}:\n"
    f"{LAUNCH_RECORD}"
)

launch = json.loads(
    LAUNCH_RECORD.read_text(
        encoding="utf-8"
    )
)

assert int(launch["run_seed"]) == RUN_SEED

RUN_DIR = Path(launch["run_dir"])

MANIFEST = RUN_DIR / "run_manifest.json"
METRICS = RUN_DIR / "metrics.jsonl"

assert RUN_DIR.is_dir(), RUN_DIR
assert MANIFEST.is_file(), MANIFEST

manifest = json.loads(
    MANIFEST.read_text(
        encoding="utf-8"
    )
)

# ------------------------------------------------------------
# 2. Verificar identidad
# ------------------------------------------------------------

manifest_seed = (
    manifest
    .get("configuration", {})
    .get("seed")
)

assert manifest_seed == RUN_SEED, (
    f"Seed del manifest={manifest_seed}, "
    f"pero RUN_SEED={RUN_SEED}"
)

architecture = manifest.get("architecture")
status = manifest.get("status")
stage = (
    manifest
    .get("lifecycle", {})
    .get("stage")
)

print("RUN_DIR      =", RUN_DIR)
print("architecture =", architecture)
print("run_seed     =", manifest_seed)
print("status       =", status)
print("stage        =", stage)

# ------------------------------------------------------------
# 3. ¿Sigue existiendo el trainer?
# ------------------------------------------------------------

ps = subprocess.run(
    ["ps", "-eo", "pid,stat,cmd"],
    capture_output=True,
    text=True,
    check=True,
)

matching_trainers = []

for line in ps.stdout.splitlines():

    if "scripts/train.py" not in line:
        continue

    if "grep" in line:
        continue

    # La config runtime lleva la seed en el nombre.
    if f"seed{RUN_SEED}_runtime.yaml" in line:
        matching_trainers.append(line)

print()
print("ACTIVE TRAINERS FOR THIS SEED =", len(matching_trainers))

for line in matching_trainers:
    print(line)

# ------------------------------------------------------------
# 4. Leer métricas
# ------------------------------------------------------------

rows = []

if METRICS.is_file():

    for line in METRICS.read_text(
        encoding="utf-8"
    ).splitlines():

        if line.strip():
            rows.append(json.loads(line))

epochs_completed = len(rows)

print()
print("=" * 90)
print("PROGRESO")
print("=" * 90)

print("epochs_completed =", epochs_completed)

if not rows:

    print()
    print("ETA_NOT_READY")
    print("Todavía no hay épocas completas en metrics.jsonl.")
    raise SystemExit

latest_epoch = int(rows[-1]["epoch"])

# ------------------------------------------------------------
# 5. Comprobar finitud
# ------------------------------------------------------------

nonfinite_epochs = []

for row in rows:

    train_loss = float(
        row["train"]["mean_loss"]
    )

    val_loss = float(
        row["validation"]["mean_loss"]
    )

    if not (
        math.isfinite(train_loss)
        and math.isfinite(val_loss)
    ):
        nonfinite_epochs.append(
            int(row["epoch"])
        )

print("nonfinite_epochs =", nonfinite_epochs)

if nonfinite_epochs:
    print()
    print("ETA_ABORTED_NONFINITE_METRICS")
    raise SystemExit

# ------------------------------------------------------------
# 6. Losses y mejor época
# ------------------------------------------------------------

val_losses = [
    float(
        row["validation"]["mean_loss"]
    )
    for row in rows
]

best_idx = min(
    range(len(val_losses)),
    key=val_losses.__getitem__
)

best_epoch = int(
    rows[best_idx]["epoch"]
)

best_val = float(
    val_losses[best_idx]
)

latest_val = float(
    val_losses[-1]
)

latest_train = float(
    rows[-1]["train"]["mean_loss"]
)

bad_epochs_so_far = (
    latest_epoch - best_epoch
)

# ------------------------------------------------------------
# 7. Patience
# ------------------------------------------------------------

early_cfg = (
    manifest
    .get("training", {})
    .get("early_stopping", {})
)

patience = int(
    early_cfg.get(
        "patience",
        DEFAULT_PATIENCE
    )
)

# ------------------------------------------------------------
# 8. Timing real
# ------------------------------------------------------------

started_raw = manifest.get(
    "started_at_utc"
)

if not started_raw:

    started_raw = (
        manifest
        .get("lifecycle", {})
        .get("started_at_utc")
    )

if not started_raw:

    started_raw = launch.get(
        "launched_at_utc"
    )

assert started_raw, (
    "No encuentro timestamp de inicio."
)

started = datetime.fromisoformat(
    started_raw.replace(
        "Z",
        "+00:00"
    )
)

now = datetime.now(
    timezone.utc
)

elapsed_seconds = max(
    0.0,
    (now - started).total_seconds()
)

avg_seconds_per_epoch = (
    elapsed_seconds
    / max(1, epochs_completed)
)

# ------------------------------------------------------------
# 9. Helpers
# ------------------------------------------------------------

def fmt_duration(seconds):

    seconds = max(
        0,
        int(seconds)
    )

    hours, rem = divmod(
        seconds,
        3600
    )

    minutes, seconds = divmod(
        rem,
        60
    )

    if hours:
        return (
            f"{hours} h {minutes} min"
        )

    if minutes:
        return (
            f"{minutes} min {seconds} s"
        )

    return f"{seconds} s"


def eta_clock(seconds):

    target = (
        now.timestamp()
        + max(0, seconds)
    )

    return datetime.fromtimestamp(
        target,
        tz=timezone.utc
    ).astimezone().strftime(
        "%H:%M"
    )

# ------------------------------------------------------------
# 10. Esperar a >= época 10
# ------------------------------------------------------------

print()
print("=" * 90)
print("ÚLTIMA ÉPOCA")
print("=" * 90)

print(
    f"epoch={latest_epoch} "
    f"train={latest_train:.6f} "
    f"val={latest_val:.6f}"
)

print(
    "best_epoch    =",
    best_epoch
)

print(
    "best_val_loss =",
    round(best_val, 6)
)

print(
    "bad_epochs    =",
    bad_epochs_so_far,
)

print(
    "patience      =",
    patience
)

if epochs_completed < MIN_EPOCHS_FOR_ETA:

    print()
    print(
        f"ETA_NOT_READY: "
        f"esperar hasta >= época "
        f"{MIN_EPOCHS_FOR_ETA}."
    )

    raise SystemExit

# ------------------------------------------------------------
# 11. ETA máxima hasta epoch 100
# ------------------------------------------------------------

remaining_to_max = max(
    0,
    MAX_EPOCHS - latest_epoch
)

eta_max_seconds = (
    remaining_to_max
    * avg_seconds_per_epoch
)

# ------------------------------------------------------------
# 12. ETA condicional de early stopping
# ------------------------------------------------------------

projected_stop_epoch = (
    best_epoch + patience
)

early_stop_remaining = max(
    0,
    projected_stop_epoch
    - latest_epoch
)

eta_early_seconds = (
    early_stop_remaining
    * avg_seconds_per_epoch
)

# ------------------------------------------------------------
# 13. Evolución reciente
# ------------------------------------------------------------

recent_n = min(
    5,
    len(rows)
)

recent_vals = (
    val_losses[-recent_n:]
)

# ¿El mejor global está entre las últimas 5?
best_is_recent = (
    best_epoch
    >= latest_epoch - recent_n + 1
)

# ------------------------------------------------------------
# 14. Timing
# ------------------------------------------------------------

print()
print("=" * 90)
print("TIMING")
print("=" * 90)

print(
    "elapsed       =",
    fmt_duration(
        elapsed_seconds
    )
)

print(
    "avg_per_epoch =",
    fmt_duration(
        avg_seconds_per_epoch
    )
)

print()
print(
    f"ETA máxima hasta epoch {MAX_EPOCHS}:"
)

print(
    "remaining =",
    fmt_duration(
        eta_max_seconds
    )
)

print(
    "hora aprox =",
    eta_clock(
        eta_max_seconds
    )
)

# ------------------------------------------------------------
# 15. Early stopping
# ------------------------------------------------------------

print()
print("=" * 90)
print("EARLY STOPPING")
print("=" * 90)

print(
    "recent_val_losses =",
    [
        round(value, 6)
        for value in recent_vals
    ]
)

if status == "completed":

    print()
    print("TRAINING_ALREADY_COMPLETED")

    training = manifest.get(
        "training",
        {}
    )

    print(
        "epochs_completed =",
        training.get(
            "epochs_completed",
            latest_epoch
        )
    )

    print(
        "stopped_early =",
        training.get(
            "stopped_early"
        )
    )

    print(
        "stop_reason =",
        training.get(
            "stop_reason"
        )
    )

elif bad_epochs_so_far == 0:

    print()
    print(
        "La última época es actualmente "
        "la mejor."
    )

    print(
        "EARLY_STOP_ETA="
        "NO_ESTIMABLE_TODAVIA"
    )

    print(
        "Una nueva mejor validation_loss "
        "reinicia el contador."
    )

else:

    print()
    print(
        "Si NO vuelve a aparecer "
        "una mejor validation_loss:"
    )

    print(
        "projected_stop_epoch =",
        projected_stop_epoch
    )

    print(
        "remaining            =",
        fmt_duration(
            eta_early_seconds
        )
    )

    print(
        "hora aprox           =",
        eta_clock(
            eta_early_seconds
        )
    )

    if best_is_recent:

        print()
        print(
            "AVISO: el mejor valor es reciente; "
            "esta ETA es todavía inestable."
        )

# ------------------------------------------------------------
# 16. Resumen final
# ------------------------------------------------------------

print()
print("=" * 90)
print(
    f"SEED {RUN_SEED} · RESUMEN ETA"
)
print("=" * 90)

print(
    f"Época actual       : "
    f"{latest_epoch}/{MAX_EPOCHS}"
)

print(
    f"Ritmo medio        : "
    f"{fmt_duration(avg_seconds_per_epoch)}/época"
)

print(
    f"Mejor época        : "
    f"{best_epoch}"
)

print(
    f"Mejor val loss     : "
    f"{best_val:.6f}"
)

print(
    f"Sin mejorar        : "
    f"{bad_epochs_so_far}/{patience} épocas"
)

print(
    f"Máximo restante    : "
    f"{fmt_duration(eta_max_seconds)}"
)

if (
    status == "running"
    and bad_epochs_so_far > 0
):

    print(
        f"Early-stop posible : "
        f"epoch {projected_stop_epoch}"
    )

    print(
        f"ETA condicional    : "
        f"{fmt_duration(eta_early_seconds)}"
    )

print()
print(
    "IMPORTANTE: la predicción de early stopping "
    "es condicional. Una nueva mejor validation_loss "
    "reinicia el contador."
)

print("=" * 90)

MODELO A · SEED 23 · ETA CHECK
RUN_DIR      = /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9/model_a_nodal_multiscale_pair/run_20260903T081000.855970Z-4995095d
architecture = model_a_nodal_multiscale_pair
run_seed     = 23
status       = completed
stage        = finalizing

ACTIVE TRAINERS FOR THIS SEED = 0

PROGRESO
epochs_completed = 52
nonfinite_epochs = []

ÚLTIMA ÉPOCA
epoch=52 train=0.261026 val=0.257980
best_epoch    = 37
best_val_loss = 0.204944
bad_epochs    = 15
patience      = 15

TIMING
elapsed       = 1 h 28 min
avg_per_epoch = 1 min 41 s

ETA máxima hasta epoch 100:
remaining = 1 h 21 min
hora aprox = 10:59

EARLY STOPPING
recent_val_losses = [0.251958, 0.236514, 0.253684, 0.263754, 0.25798]

TRAINING_ALREADY_COMPLETED
epochs_completed = 52
stopped_early = True
stop_reason = early_stopping

SEED 23 · RESUMEN ETA
Época actual       : 52/100
Ritmo medio        : 1 min 41 s/época
Mejor época        : 37
Mejor val loss     : 0.204944
Sin mejorar        :

## Evaluacion de estado de entrenamiento DESPUES DE PERDER CONEXION

In [ ]:
from pathlib import Path
import json
import subprocess

# ============================================================
# MODELO A · A9
# RECUPERACIÓN TRAS CIERRE DE COLAB
# SOLO LECTURA
# ============================================================

A9_ROOT = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_a/runs/model_a_a9"
)

ARCHITECTURE = "model_a_nodal_multiscale_pair"

ARCH_ROOT = A9_ROOT / ARCHITECTURE
QUEUE_DIR = A9_ROOT / "_queue"

QUEUE_STATE = (
    QUEUE_DIR
    / "model_a_a9_multiseed_queue_state.json"
)

QUEUE_LAUNCHER = (
    QUEUE_DIR
    / "model_a_a9_multiseed_queue_launcher.json"
)

QUEUE_LOG = (
    QUEUE_DIR
    / "model_a_a9_multiseed_queue.log"
)

SEEDS = [11, 23, 37, 41, 53]


def read_json(path):
    try:
        return json.loads(
            path.read_text(encoding="utf-8")
        )
    except Exception as exc:
        return {
            "_read_error": f"{type(exc).__name__}: {exc}"
        }


print("=" * 100)
print("A9 MODEL A · RECOVERY STATUS")
print("=" * 100)

print("A9_ROOT exists =", A9_ROOT.exists())
print("ARCH_ROOT exists =", ARCH_ROOT.exists())

# ============================================================
# 1. ¿Seguimos en el mismo runtime?
# ============================================================

print()
print("=" * 100)
print("RUNTIME LOCAL")
print("=" * 100)

LOCAL_WORKSPACE = Path(
    "/content/model_a_workspace"
)

print(
    "/content/model_a_workspace exists =",
    LOCAL_WORKSPACE.exists()
)

print(
    "repo exists =",
    (
        LOCAL_WORKSPACE / "repo"
    ).exists()
)

print(
    "staging exists =",
    (
        LOCAL_WORKSPACE / "staging"
    ).exists()
)

# ============================================================
# 2. Procesos actualmente vivos
# ============================================================

ps = subprocess.run(
    ["ps", "-eo", "pid,stat,etime,cmd"],
    capture_output=True,
    text=True,
    check=True,
)

supervisors = []
trainers = []

for line in ps.stdout.splitlines():

    if (
        "a9_model_a_multiseed_queue_supervisor.py"
        in line
        and "grep" not in line
    ):
        supervisors.append(line)

    if (
        "scripts/train.py" in line
        and "grep" not in line
    ):
        trainers.append(line)

print()
print("=" * 100)
print("PROCESOS ACTIVOS")
print("=" * 100)

print("QUEUE_SUPERVISORS =", len(supervisors))

for line in supervisors:
    print(" ", line)

print()
print("TRAINERS =", len(trainers))

for line in trainers:
    print(" ", line)

# ============================================================
# 3. Estado persistente de la cola
# ============================================================

print()
print("=" * 100)
print("QUEUE STATE EN DRIVE")
print("=" * 100)

print(
    "QUEUE_STATE exists =",
    QUEUE_STATE.exists()
)

queue_state = None

if QUEUE_STATE.is_file():

    queue_state = read_json(
        QUEUE_STATE
    )

    for key in [
        "status",
        "current_seed",
        "current_run_dir",
        "current_manifest_status",
        "current_stage",
        "current_epochs_completed",
        "current_metric_records",
        "completed_seeds",
        "updated_at_utc",
        "completed_at_utc",
        "error",
    ]:

        if key in queue_state:
            print(
                f"{key} =",
                queue_state.get(key)
            )

print()
print(
    "QUEUE_LAUNCHER exists =",
    QUEUE_LAUNCHER.exists()
)

if QUEUE_LAUNCHER.is_file():

    launcher = read_json(
        QUEUE_LAUNCHER
    )

    print(
        "old_queue_pid =",
        launcher.get("pid")
    )

    print(
        "launched_at_utc =",
        launcher.get(
            "launched_at_utc"
        )
    )

# ============================================================
# 4. Inventario completo de runs por seed
# ============================================================

print()
print("=" * 100)
print("RUNS PERSISTIDOS EN DRIVE")
print("=" * 100)

seed_status = {}

for seed in SEEDS:

    print()
    print("-" * 100)
    print(f"SEED {seed}")
    print("-" * 100)

    runs = []

    if ARCH_ROOT.exists():

        for run_dir in sorted(
            ARCH_ROOT.glob("run_*")
        ):

            manifest_path = (
                run_dir
                / "run_manifest.json"
            )

            if not manifest_path.is_file():
                continue

            manifest = read_json(
                manifest_path
            )

            manifest_seed = (
                manifest
                .get("configuration", {})
                .get("seed")
            )

            if manifest_seed != seed:
                continue

            runs.append(
                (
                    run_dir,
                    manifest,
                )
            )

    print("num_runs =", len(runs))

    seed_status[seed] = []

    for run_dir, manifest in runs:

        training = manifest.get(
            "training",
            {}
        )

        status = manifest.get(
            "status"
        )

        stage = (
            manifest
            .get("lifecycle", {})
            .get("stage")
        )

        metrics_path = (
            run_dir / "metrics.jsonl"
        )

        metric_records = 0
        last_metric_epoch = None

        if metrics_path.is_file():

            lines = [
                line
                for line
                in metrics_path.read_text(
                    encoding="utf-8",
                    errors="replace",
                ).splitlines()
                if line.strip()
            ]

            metric_records = len(lines)

            if lines:

                try:
                    latest = json.loads(
                        lines[-1]
                    )

                    last_metric_epoch = (
                        latest.get("epoch")
                    )

                except Exception:
                    pass

        best_pt = (
            run_dir
            / "checkpoints"
            / "best.pt"
        )

        last_pt = (
            run_dir
            / "checkpoints"
            / "last.pt"
        )

        acceptance_path = (
            run_dir
            / "a9_acceptance.json"
        )

        acceptance = None

        if acceptance_path.is_file():
            acceptance = read_json(
                acceptance_path
            )

        print()
        print(run_dir.name)

        print(
            "  status           =",
            status
        )

        print(
            "  stage            =",
            stage
        )

        print(
            "  epochs_completed =",
            training.get(
                "epochs_completed"
            )
        )

        print(
            "  global_step      =",
            training.get(
                "global_step"
            )
        )

        print(
            "  metric_records   =",
            metric_records
        )

        print(
            "  last_metric_epoch=",
            last_metric_epoch
        )

        print(
            "  best.pt          =",
            best_pt.is_file()
        )

        print(
            "  last.pt          =",
            last_pt.is_file()
        )

        if acceptance is not None:

            print(
                "  acceptance       =",
                acceptance.get("status")
            )

        seed_status[seed].append(
            {
                "run_dir": str(run_dir),
                "status": status,
                "epochs":
                    training.get(
                        "epochs_completed"
                    ),
                "metrics":
                    metric_records,
                "best_pt":
                    best_pt.is_file(),
                "last_pt":
                    last_pt.is_file(),
            }
        )

# ============================================================
# 5. Últimas líneas del log de la cola
# ============================================================

print()
print("=" * 100)
print("QUEUE LOG · ÚLTIMAS 60 LÍNEAS")
print("=" * 100)

if QUEUE_LOG.is_file():

    lines = QUEUE_LOG.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines()

    for line in lines[-60:]:
        print(line)

else:
    print("QUEUE_LOG no existe.")

# ============================================================
# 6. Clasificación automática
# ============================================================

print()
print("=" * 100)
print("DIAGNÓSTICO")
print("=" * 100)

productive_seeds = [23, 37, 41, 53]

running_runs = []

failed_or_interrupted = []

completed_by_seed = {}

for seed in productive_seeds:

    completed_by_seed[seed] = False

    for item in seed_status.get(
        seed,
        []
    ):

        if item["status"] == "completed":
            completed_by_seed[seed] = True

        elif item["status"] == "running":
            running_runs.append(
                (seed, item)
            )

        elif item["status"] in {
            "failed",
            "interrupted",
        }:

            failed_or_interrupted.append(
                (seed, item)
            )

if supervisors or trainers:

    print(
        "RECOVERY_CLASS="
        "LIVE_PROCESSES_PRESENT"
    )

    print(
        "El runtime aún contiene procesos."
    )

    print(
        "NO relanzar entrenamiento ni cola."
    )

elif (
    queue_state is not None
    and queue_state.get("status")
    == "training_queue_completed"
):

    print(
        "RECOVERY_CLASS="
        "TRAINING_QUEUE_COMPLETE"
    )

    print(
        "Los entrenamientos ya terminaron."
    )

    print(
        "NEXT=auditoría/export/acceptance."
    )

elif running_runs:

    print(
        "RECOVERY_CLASS="
        "STALE_RUNNING_RUN_AFTER_RUNTIME_LOSS"
    )

    print()
    print(
        "Hay un run cuyo manifest sigue "
        "en 'running', pero no existe "
        "ningún proceso train.py."
    )

    for seed, item in running_runs:

        print(
            f"  seed={seed} "
            f"epochs={item['epochs']} "
            f"last.pt={item['last_pt']}"
        )

    print()
    print(
        "NO hacer fresh."
    )

    print(
        "Será necesario RESUME desde last.pt."
    )

elif failed_or_interrupted:

    print(
        "RECOVERY_CLASS="
        "INTERRUPTED_OR_FAILED_RUN"
    )

    for seed, item in (
        failed_or_interrupted
    ):

        print(
            f"  seed={seed} "
            f"status={item['status']} "
            f"epochs={item['epochs']} "
            f"last.pt={item['last_pt']}"
        )

    print()
    print(
        "NO hacer fresh."
    )

    print(
        "Hay que decidir resume/diagnóstico."
    )

else:

    completed_prefix = []

    for seed in productive_seeds:

        if completed_by_seed[seed]:
            completed_prefix.append(seed)
        else:
            break

    remaining = [
        seed
        for seed in productive_seeds
        if not completed_by_seed[seed]
    ]

    print(
        "RECOVERY_CLASS="
        "NO_ACTIVE_INTERRUPTED_RUN"
    )

    print(
        "completed_seeds =",
        completed_prefix
    )

    print(
        "remaining_seeds =",
        remaining
    )

    print()
    print(
        "Tras reconstruir el entorno, "
        "podremos reactivar la cola "
        "desde la siguiente seed pendiente."
    )

print("=" * 100)

A9 MODEL A · RECOVERY STATUS
A9_ROOT exists = True
ARCH_ROOT exists = True

RUNTIME LOCAL
/content/model_a_workspace exists = True
repo exists = True
staging exists = True

PROCESOS ACTIVOS
QUEUE_SUPERVISORS = 0

TRAINERS = 0

QUEUE STATE EN DRIVE
QUEUE_STATE exists = True
status = training_queue_completed
current_seed = None
current_run_dir = None
current_manifest_status = completed
current_stage = finalizing
current_epochs_completed = 37
current_metric_records = 37
completed_seeds = [23, 37, 41, 53]
updated_at_utc = 2026-09-03T10:37:00.714264+00:00
completed_at_utc = 2026-09-03T10:37:00.714232+00:00

QUEUE_LAUNCHER exists = True
old_queue_pid = 7515
launched_at_utc = 2026-09-03T08:30:36.907484+00:00

RUNS PERSISTIDOS EN DRIVE

----------------------------------------------------------------------------------------------------
SEED 11
----------------------------------------------------------------------------------------------------
num_runs = 2

run_20260902T093129.973812Z-5c337698


# **A11 seed_training**

In [ ]:
from pathlib import Path
import json
import os
import sys

# ============================================================
# A9 · MODELO A · SEED 11
# Persistencia del preflight + config runtime
# ============================================================

REPO_DIR = Path("/content/model_a_workspace/repo")

MUTANTS_HDF5 = Path(
    "/content/model_a_workspace/staging/proc_483p.hdf5"
)

WT_HDF5 = Path(
    "/content/model_a_workspace/staging/wt_companion.hdf5"
)

A_OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_a/runs/model_a_a9"
)

# IMPORTANTE:
# debe ser EXACTAMENTE el peer output root que usaste
# en el A9_PREFLIGHT que ya dio PASS.
B_OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_b/runs/model_b_a9"
)

ALLOWED_OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2"
)

EXPECTED_COMMIT = (
    "3757bfdde52a9920611f3e7b8027434ce80d54bc"
)

RUN_SEED = 11

ARCHITECTURE = "model_a_nodal_multiscale_pair"

CONFIG_A9 = REPO_DIR / "configs/model_a_a9.yaml"

# ------------------------------------------------------------
# Importar desde EL REPOSITORIO CLONADO
# ------------------------------------------------------------

os.chdir(REPO_DIR)

for candidate in (str(REPO_DIR), str(REPO_DIR / "src")):
    if candidate not in sys.path:
        sys.path.insert(0, candidate)

from scripts.a9_preflight import (
    validate_a9_colab_preflight,
    resolve_a9_runtime_configs,
)

from gnn_siamese.config import save_config

# ------------------------------------------------------------
# 1. Reproducir una vez el PASS únicamente para persistirlo
# ------------------------------------------------------------

preflight = validate_a9_colab_preflight(
    CONFIG_A9,
    architecture=ARCHITECTURE,
    run_seed=RUN_SEED,
    mutants_hdf5=MUTANTS_HDF5,
    wt_hdf5=WT_HDF5,
    output_root=A_OUTPUT_ROOT,
    peer_output_root=B_OUTPUT_ROOT,
    repo_root=REPO_DIR,
    expected_commit=EXPECTED_COMMIT,
    allowed_output_root=ALLOWED_OUTPUT_ROOT,
)

assert preflight["status"] == "PASS"

PREFLIGHT_DIR = A_OUTPUT_ROOT / "_preflight"
PREFLIGHT_DIR.mkdir(parents=True, exist_ok=True)

PREFLIGHT_JSON = (
    PREFLIGHT_DIR / "model_a_seed11_a9_preflight.json"
)

PREFLIGHT_JSON.write_text(
    json.dumps(preflight, indent=2, sort_keys=True, default=str),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 2. Resolver el config PRODUCTIVO real de seed 11
# ------------------------------------------------------------

config_a11, config_b11_peer = resolve_a9_runtime_configs(
    CONFIG_A9,
    architecture=ARCHITECTURE,
    run_seed=RUN_SEED,
    mutants_hdf5=MUTANTS_HDF5,
    wt_hdf5=WT_HDF5,
    output_root=A_OUTPUT_ROOT,
    peer_output_root=B_OUTPUT_ROOT,
    repo_root=REPO_DIR,
)

RUNTIME_CONFIG_DIR = Path(
    "/content/model_a_workspace/resolved_configs"
)
RUNTIME_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

RUNTIME_A11 = (
    RUNTIME_CONFIG_DIR / "model_a_a9_seed11_runtime.yaml"
)

save_config(config_a11, RUNTIME_A11)

# ------------------------------------------------------------
# 3. Comprobaciones finales antes de train
# ------------------------------------------------------------

assert config_a11["project"]["seed"] == 11
assert config_a11["split"]["seed"] == 42
assert config_a11["split"]["allow_create"] is False

assert config_a11["model"]["architecture"] == (
    "model_a_nodal_multiscale_pair"
)

assert config_a11["training"]["device"] == "cuda"
assert config_a11["training"]["epochs"] == 100
assert config_a11["training"]["batch_size"] == 4
assert config_a11["training"]["scheduler"] == "cosine"

assert config_a11["training"]["early_stopping"]["enabled"] is True
assert config_a11["training"]["early_stopping"]["patience"] == 15

assert config_a11["loss"]["main"] == "nt_xent"

assert config_a11["outputs"]["root_dir"] == str(
    A_OUTPUT_ROOT.resolve()
)

print("=" * 80)
print("A9 MODELO A · SEED 11")
print("=" * 80)

print("A9_PREFLIGHT =", preflight["status"])
print("commit       =", preflight["git"])
print("run_seed     =", config_a11["project"]["seed"])
print("split_seed   =", config_a11["split"]["seed"])
print("architecture =", config_a11["model"]["architecture"])
print("device       =", config_a11["training"]["device"])
print("epochs       =", config_a11["training"]["epochs"])
print("batch_size   =", config_a11["training"]["batch_size"])
print("scheduler    =", config_a11["training"]["scheduler"])
print(
    "early_stop  =",
    config_a11["training"]["early_stopping"],
)
print("mutants_h5   =", config_a11["paths"]["mutants_hdf5"])
print("wt_h5        =", config_a11["paths"]["wt_companion_hdf5"])
print("output_root  =", config_a11["outputs"]["root_dir"])

print()
print("PREFLIGHT_JSON =", PREFLIGHT_JSON)
print("RUNTIME_CONFIG =", RUNTIME_A11)

print()
print("READY_FOR_A11_TRAINING=YES")

A9 MODELO A · SEED 11
A9_PREFLIGHT = PASS
commit       = {'commit': '3757bfdde52a9920611f3e7b8027434ce80d54bc', 'expected_commit': '3757bfdde52a9920611f3e7b8027434ce80d54bc', 'working_tree': 'clean'}
run_seed     = 11
split_seed   = 42
architecture = model_a_nodal_multiscale_pair
device       = cuda
epochs       = 100
batch_size   = 4
scheduler    = cosine
early_stop  = {'enabled': True, 'monitor': 'validation_loss', 'mode': 'min', 'patience': 15, 'min_delta': 0.0}
mutants_h5   = /content/model_a_workspace/staging/proc_483p.hdf5
wt_h5        = /content/model_a_workspace/staging/wt_companion.hdf5
output_root  = /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9

PREFLIGHT_JSON = /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9/_preflight/model_a_seed11_a9_preflight.json
RUNTIME_CONFIG = /content/model_a_workspace/resolved_configs/model_a_a9_seed11_runtime.yaml

READY_FOR_A11_TRAINING=YES


In [ ]:
from pathlib import Path
import subprocess
import os
import sys
import time
import json

REPO_DIR = Path("/content/model_a_workspace/repo")

RUNTIME_A11 = Path(
    "/content/model_a_workspace/resolved_configs/"
    "model_a_a9_seed11_runtime.yaml"
)

A_OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_a/runs/model_a_a9"
)

MODEL_RUN_ROOT = (
    A_OUTPUT_ROOT / "model_a_nodal_multiscale_pair"
)

MODEL_RUN_ROOT.mkdir(parents=True, exist_ok=True)

# Identificar exactamente qué run crea ESTE lanzamiento
runs_before = {
    p.resolve()
    for p in MODEL_RUN_ROOT.glob("run_*")
    if p.is_dir()
}

LOG_DIR = A_OUTPUT_ROOT / "_launcher_logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_LOG = LOG_DIR / "model_a_seed11_training.log"

env = os.environ.copy()
env["PYTHONPATH"] = (
    f"{REPO_DIR / 'src'}:"
    f"{REPO_DIR}:"
    f"{env.get('PYTHONPATH', '')}"
)

cmd = [
    sys.executable,
    "-u",
    "scripts/train.py",
    "--config",
    str(RUNTIME_A11),
    "--device",
    "cuda",
]

print("Launching:")
print(" ".join(cmd))

log_handle = open(TRAIN_LOG, "w", buffering=1)

proc_a11 = subprocess.Popen(
    cmd,
    cwd=str(REPO_DIR),
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True,
)

log_handle.close()

PID_FILE = Path("/content/model_a_workspace/a11.pid")
PID_FILE.write_text(str(proc_a11.pid), encoding="utf-8")

print("PID =", proc_a11.pid)
print("LOG =", TRAIN_LOG)

# Esperar únicamente a que el trainer cree su run_dir
A11_RUN_DIR = None

for _ in range(120):
    runs_after = {
        p.resolve()
        for p in MODEL_RUN_ROOT.glob("run_*")
        if p.is_dir()
    }

    new_runs = runs_after - runs_before

    if len(new_runs) == 1:
        A11_RUN_DIR = next(iter(new_runs))
        break

    if len(new_runs) > 1:
        raise RuntimeError(
            f"Más de un run nuevo detectado: {new_runs}"
        )

    if proc_a11.poll() is not None:
        raise RuntimeError(
            f"El entrenamiento terminó prematuramente "
            f"con exit code {proc_a11.returncode}. "
            f"Revisa {TRAIN_LOG}"
        )

    time.sleep(2)

if A11_RUN_DIR is None:
    raise RuntimeError(
        "No apareció el run_dir de A11 en el tiempo esperado."
    )

Path(
    "/content/model_a_workspace/a11_run_dir.txt"
).write_text(
    str(A11_RUN_DIR),
    encoding="utf-8",
)

print()
print("=" * 80)
print("A11 STARTED")
print("=" * 80)
print("PID     =", proc_a11.pid)
print("RUN_DIR =", A11_RUN_DIR)
print("LOG     =", TRAIN_LOG)
print("A11_BACKGROUND_TRAINING=RUNNING")

Launching:
/usr/bin/python3 -u scripts/train.py --config /content/model_a_workspace/resolved_configs/model_a_a9_seed11_runtime.yaml --device cuda
PID = 3416
LOG = /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9/_launcher_logs/model_a_seed11_training.log

A11 STARTED
PID     = 3416
RUN_DIR = /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9/model_a_nodal_multiscale_pair/run_20260902T103830.451859Z-f6990a45
LOG     = /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9/_launcher_logs/model_a_seed11_training.log
A11_BACKGROUND_TRAINING=RUNNING


In [ ]:
from pathlib import Path
import json
import math
import os
import subprocess

# ============================================================
# A11 · HEALTH CHECK CORREGIDO
# NO inicia ni detiene entrenamiento
# ============================================================

RUN_DIR_FILE = Path(
    "/content/model_a_workspace/a11_run_dir.txt"
)

PID_FILE = Path(
    "/content/model_a_workspace/a11.pid"
)

assert RUN_DIR_FILE.exists(), "No existe a11_run_dir.txt"
assert PID_FILE.exists(), "No existe a11.pid"

A11_RUN_DIR = Path(
    RUN_DIR_FILE.read_text(encoding="utf-8").strip()
)

PID = int(
    PID_FILE.read_text(encoding="utf-8").strip()
)

METRICS = A11_RUN_DIR / "metrics.jsonl"
MANIFEST = A11_RUN_DIR / "run_manifest.json"
BEST = A11_RUN_DIR / "checkpoints/best.pt"
LAST = A11_RUN_DIR / "checkpoints/last.pt"

TRAIN_LOG = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_a/runs/model_a_a9/_launcher_logs/"
    "model_a_seed11_training.log"
)

print("=" * 80)
print("A11 · ESTADO DEL PROCESO")
print("=" * 80)

# ------------------------------------------------------------
# 1. ¿Sigue vivo el proceso?
# ------------------------------------------------------------

try:
    os.kill(PID, 0)
    process_alive = True
except OSError:
    process_alive = False

print("PID          =", PID)
print("process_alive =", process_alive)
print("RUN_DIR      =", A11_RUN_DIR)

# ------------------------------------------------------------
# 2. Leer metrics.jsonl
# ------------------------------------------------------------

assert METRICS.exists(), f"No existe {METRICS}"

rows = []

for line in METRICS.read_text(
    encoding="utf-8"
).splitlines():
    if line.strip():
        rows.append(json.loads(line))

assert rows, "metrics.jsonl todavía no contiene épocas."

print("metric_records =", len(rows))

# Mostrar esquema real para dejarlo documentado
print()
print("top-level keys =", sorted(rows[-1].keys()))

if "train" in rows[-1]:
    print(
        "train keys     =",
        sorted(rows[-1]["train"].keys())
    )

if "validation" in rows[-1]:
    print(
        "validation keys=",
        sorted(rows[-1]["validation"].keys())
    )

# ------------------------------------------------------------
# 3. Validación de las épocas disponibles
# ------------------------------------------------------------

previous_global_step = -1

for row in rows:

    assert "epoch" in row
    assert "global_step" in row
    assert "train" in row
    assert "validation" in row

    train_loss = float(
        row["train"]["mean_loss"]
    )

    validation_loss = float(
        row["validation"]["mean_loss"]
    )

    assert math.isfinite(train_loss), (
        f"train mean_loss no finita en epoch "
        f"{row['epoch']}: {train_loss}"
    )

    assert math.isfinite(validation_loss), (
        f"validation mean_loss no finita en epoch "
        f"{row['epoch']}: {validation_loss}"
    )

    global_step = int(row["global_step"])

    assert global_step > previous_global_step, (
        "global_step no avanza de forma estricta"
    )

    previous_global_step = global_step

print()
print("=" * 80)
print("PRIMERAS ÉPOCAS")
print("=" * 80)

for row in rows[:3]:
    print(
        f"epoch={row['epoch']} | "
        f"global_step={row['global_step']} | "
        f"train_loss={row['train']['mean_loss']:.6f} | "
        f"validation_loss="
        f"{row['validation']['mean_loss']:.6f}"
    )

print()
print("=" * 80)
print("ÚLTIMAS ÉPOCAS DISPONIBLES")
print("=" * 80)

for row in rows[-3:]:
    print(
        f"epoch={row['epoch']} | "
        f"global_step={row['global_step']} | "
        f"train_loss={row['train']['mean_loss']:.6f} | "
        f"validation_loss="
        f"{row['validation']['mean_loss']:.6f}"
    )

# ------------------------------------------------------------
# 4. Manifest
# ------------------------------------------------------------

assert MANIFEST.exists(), f"No existe {MANIFEST}"

manifest = json.loads(
    MANIFEST.read_text(encoding="utf-8")
)

print()
print("=" * 80)
print("MANIFEST")
print("=" * 80)

print(
    "status       =",
    manifest.get("status")
)

print(
    "architecture =",
    manifest.get("architecture")
)

configuration = manifest.get(
    "configuration", {}
)

print(
    "run_seed     =",
    configuration.get("seed")
)

print(
    "split_seed   =",
    configuration.get(
        "seed_bundle", {}
    ).get("split")
)

training = manifest.get(
    "training", {}
)

print(
    "epochs_planned   =",
    training.get("epochs_planned")
)

print(
    "epochs_completed =",
    training.get("epochs_completed")
)

print(
    "global_step      =",
    training.get("global_step")
)

print(
    "best_metric      =",
    training.get("best_metric")
)

print(
    "stopped_early    =",
    training.get("stopped_early")
)

print(
    "stop_reason      =",
    training.get("stop_reason")
)

# ------------------------------------------------------------
# 5. Checkpoints
# ------------------------------------------------------------

print()
print("=" * 80)
print("CHECKPOINTS")
print("=" * 80)

print(
    "best.pt =",
    BEST.exists(),
    BEST.stat().st_size if BEST.exists() else None
)

print(
    "last.pt =",
    LAST.exists(),
    LAST.stat().st_size if LAST.exists() else None
)

assert BEST.exists() and BEST.stat().st_size > 0
assert LAST.exists() and LAST.stat().st_size > 0

# ------------------------------------------------------------
# 6. Últimas líneas del log
# ------------------------------------------------------------

print()
print("=" * 80)
print("ÚLTIMAS LÍNEAS DEL LOG")
print("=" * 80)

if TRAIN_LOG.exists():

    log_lines = TRAIN_LOG.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    for line in log_lines[-30:]:
        print(line)

else:
    print("TRAIN_LOG no encontrado:", TRAIN_LOG)

# ------------------------------------------------------------
# 7. GPU
# ------------------------------------------------------------

print()
print("=" * 80)
print("GPU")
print("=" * 80)

subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu="
        "name,memory.used,memory.total,utilization.gpu",
        "--format=csv,noheader",
    ],
    check=False,
)

# ------------------------------------------------------------
# 8. Resultado
# ------------------------------------------------------------

print()
print("=" * 80)

if process_alive:

    print("A11_EARLY_HEALTH_CHECK=PASS")
    print(
        "A11 continúa entrenándose. NO reiniciar."
    )

elif manifest.get("status") == "completed":

    print("A11_PROCESS_FINISHED")
    print(
        "El proceso ya terminó correctamente; "
        "pasaremos directamente a acceptance."
    )

else:

    print("A11_PROCESS_NOT_RUNNING")
    print(
        "NO relanzar todavía. Hay que revisar "
        "manifest y log para conocer la causa."
    )

print("=" * 80)

AssertionError: No existe a11_run_dir.txt

In [ ]:
from pathlib import Path
import json
import math

# ============================================================
# A11 ORIGINAL · AUDITORÍA POST-TRAINING
# ============================================================

A11 = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_a/runs/model_a_a9/"
    "model_a_nodal_multiscale_pair/"
    "run_20260902T093129.973812Z-5c337698"
)

MANIFEST = A11 / "run_manifest.json"
METRICS = A11 / "metrics.jsonl"
GRADIENTS = A11 / "gradient_audit.json"
BEST = A11 / "checkpoints/best.pt"
LAST = A11 / "checkpoints/last.pt"
CONFIG = A11 / "config_resolved.yaml"
SPLIT = A11 / "split.json"

print("=" * 90)
print("A11 ORIGINAL · POST-TRAINING AUDIT")
print("=" * 90)

required = {
    "run_manifest.json": MANIFEST,
    "metrics.jsonl": METRICS,
    "gradient_audit.json": GRADIENTS,
    "best.pt": BEST,
    "last.pt": LAST,
    "config_resolved.yaml": CONFIG,
    "split.json": SPLIT,
}

for name, path in required.items():

    exists = path.exists()
    size = path.stat().st_size if exists else 0

    print(
        f"{name:24s} exists={exists} size={size}"
    )

    assert exists
    assert size > 0


# ============================================================
# MANIFEST
# ============================================================

manifest = json.loads(
    MANIFEST.read_text(encoding="utf-8")
)

print()
print("=" * 90)
print("MANIFEST")
print("=" * 90)

training = manifest.get("training", {})
configuration = manifest.get("configuration", {})
data = manifest.get("data", {})

print("status            =", manifest.get("status"))
print("architecture      =", manifest.get("architecture"))

print(
    "run_seed          =",
    configuration.get("seed")
)

print(
    "split_seed        =",
    configuration.get(
        "seed_bundle", {}
    ).get("split")
)

print(
    "epochs_planned    =",
    training.get("epochs_planned")
)

print(
    "epochs_completed  =",
    training.get("epochs_completed")
)

print(
    "stopped_early     =",
    training.get("stopped_early")
)

print(
    "stop_reason       =",
    training.get("stop_reason")
)

print(
    "best_metric       =",
    training.get("best_metric")
)

print(
    "early_best_metric =",
    training.get(
        "early_stopping", {}
    ).get("best_metric")
)

print(
    "bad_epochs        =",
    training.get(
        "early_stopping", {}
    ).get("bad_epochs")
)

print(
    "global_step       =",
    training.get("global_step")
)

print(
    "optimizer         =",
    training.get("optimizer")
)

print(
    "scheduler         =",
    training.get(
        "scheduler_details", {}
    ).get("class")
)

print(
    "batch_size        =",
    training.get("batch_size")
)

print(
    "learning_rate     =",
    training.get("learning_rate")
)

print(
    "weight_decay      =",
    training.get("weight_decay")
)

assert manifest.get("status") == "completed"

assert manifest.get("architecture") == (
    "model_a_nodal_multiscale_pair"
)

assert configuration.get("seed") == 11

assert (
    configuration.get(
        "seed_bundle", {}
    ).get("split")
    == 42
)

assert training.get("epochs_planned") == 100

assert 1 <= int(
    training.get("epochs_completed")
) <= 100


# ============================================================
# METRICS
# ============================================================

rows = [
    json.loads(line)
    for line in METRICS.read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]

print()
print("=" * 90)
print("METRICS")
print("=" * 90)

print("metric_records =", len(rows))

assert len(rows) == int(
    training["epochs_completed"]
)

nonfinite = []

for row in rows:

    train_loss = float(
        row["train"]["mean_loss"]
    )

    validation_loss = float(
        row["validation"]["mean_loss"]
    )

    if (
        not math.isfinite(train_loss)
        or not math.isfinite(validation_loss)
    ):
        nonfinite.append(row["epoch"])

print("nonfinite_epochs =", nonfinite)

assert not nonfinite

print()
print("PRIMERAS 3 ÉPOCAS")

for row in rows[:3]:

    print(
        f"epoch={row['epoch']} "
        f"step={row['global_step']} "
        f"train={row['train']['mean_loss']:.6f} "
        f"val={row['validation']['mean_loss']:.6f}"
    )

print()
print("ÚLTIMAS 5 ÉPOCAS")

for row in rows[-5:]:

    print(
        f"epoch={row['epoch']} "
        f"step={row['global_step']} "
        f"train={row['train']['mean_loss']:.6f} "
        f"val={row['validation']['mean_loss']:.6f}"
    )


# ============================================================
# MEJOR VALIDATION LOSS SEGÚN METRICS
# ============================================================

best_row = min(
    rows,
    key=lambda row:
        float(row["validation"]["mean_loss"])
)

print()
print("=" * 90)
print("BEST VALIDATION")
print("=" * 90)

print(
    "best_epoch          =",
    best_row["epoch"]
)

print(
    "best_validation_loss=",
    best_row["validation"]["mean_loss"]
)

print(
    "train_at_best       =",
    best_row["train"]["mean_loss"]
)


# ============================================================
# GRADIENT AUDIT
# ============================================================

grad = json.loads(
    GRADIENTS.read_text(encoding="utf-8")
)

print()
print("=" * 90)
print("GRADIENT AUDIT")
print("=" * 90)

expected_modules = [
    "encoder",
    "node_delta_block",
    "pair_fusion",
    "projection_pair_a",
]

for name in expected_modules:

    assert name in grad, (
        f"Falta módulo esperado: {name}"
    )

    rec = grad[name]

    print()
    print(name)
    print("  status                 =", rec.get("status"))
    print(
        "  optimizer_group        =",
        rec.get("optimizer_group")
    )
    print(
        "  mean_gradient_norm     =",
        rec.get("mean_gradient_norm")
    )
    print(
        "  max_gradient_norm      =",
        rec.get("max_gradient_norm")
    )
    print(
        "  none_gradient_fraction =",
        rec.get("none_gradient_fraction")
    )
    print(
        "  zero_gradient_fraction =",
        rec.get("zero_gradient_fraction")
    )
    print(
        "  has_nan_or_inf         =",
        rec.get("has_nan_or_inf")
    )
    print(
        "  relative_weight_change =",
        rec.get("relative_weight_change")
    )


# ============================================================
# DATASET IDENTITY
# ============================================================

print()
print("=" * 90)
print("DATASET / SPLIT")
print("=" * 90)

inventory = data.get("inventory", {})

print(
    "biological_variants =",
    inventory.get("biological_variants")
)

print(
    "native_wt_controls  =",
    inventory.get("native_wt_controls")
)

print(
    "split_fingerprint   =",
    data.get("split_fingerprint")
)

hdf5_fp = data.get(
    "hdf5_content_fingerprint", {}
)

print(
    "combined_hdf5_digest =",
    hdf5_fp.get(
        "combined", {}
    ).get("digest")
)

print()
print("=" * 90)
print("A11_POST_TRAINING_BASIC_AUDIT=PASS")
print("=" * 90)

A11 ORIGINAL · POST-TRAINING AUDIT
run_manifest.json        exists=True size=40220
metrics.jsonl            exists=True size=36342
gradient_audit.json      exists=True size=2526
best.pt                  exists=True size=85860777
last.pt                  exists=True size=85860777
config_resolved.yaml     exists=True size=15223
split.json               exists=True size=99409

MANIFEST
status            = completed
architecture      = model_a_nodal_multiscale_pair
run_seed          = 11
split_seed        = 42
epochs_planned    = 100
epochs_completed  = 47
stopped_early     = True
stop_reason       = early_stopping
best_metric       = 0.21073842965639555
early_best_metric = 0.21073842965639555
bad_epochs        = 15
global_step       = 4042
optimizer         = adamw
scheduler         = CosineAnnealingLR
batch_size        = 4
learning_rate     = 0.001
weight_decay      = 0.0001

METRICS
metric_records = 47
nonfinite_epochs = []

PRIMERAS 3 ÉPOCAS
epoch=1 step=86 train=1.084877 val=0.669614


In [ ]:
from pathlib import Path
import os
import sys
import json
import numpy as np
import torch

from torch.utils.data import DataLoader

# ============================================================
# A11 · EXPORTACIÓN DE REPRESENTACIONES DESDE best.pt
# Evaluación determinista sobre las 483 variantes biológicas
# ============================================================

REPO_DIR = Path("/content/model_a_workspace/repo")

A11 = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_a/runs/model_a_a9/"
    "model_a_nodal_multiscale_pair/"
    "run_20260902T093129.973812Z-5c337698"
)

CONFIG_PATH = A11 / "config_resolved.yaml"
BEST = A11 / "checkpoints/best.pt"

EXPORT_DIR = A11 / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

EXPORT_NPZ = (
    EXPORT_DIR
    / "a9_pair_representations_best.npz"
)

VARIANT_IDS_TXT = (
    EXPORT_DIR
    / "a9_pair_representations_variant_ids.txt"
)

# ------------------------------------------------------------
# Entorno repo
# ------------------------------------------------------------

os.chdir(REPO_DIR)

for candidate in (
    str(REPO_DIR),
    str(REPO_DIR / "src"),
):
    if candidate not in sys.path:
        sys.path.insert(0, candidate)

from gnn_siamese.config import load_config
from gnn_siamese.builders import build_training_pipeline
from gnn_siamese.data import collate_mut_wt_pairs
from gnn_siamese.models.multiscale_pooling_a import (
    indices_to_mask,
    aligned_selection_mask,
)
from gnn_siamese.training.checkpointing import (
    load_checkpoint,
)

# ------------------------------------------------------------
# 1. Reconstruir exactamente el pipeline del run
# ------------------------------------------------------------

config = load_config(CONFIG_PATH)

config["__config_path__"] = str(
    CONFIG_PATH.resolve()
)

assert config["model"]["architecture"] == (
    "model_a_nodal_multiscale_pair"
)

assert config["project"]["seed"] == 11
assert config["split"]["seed"] == 42

pipeline = build_training_pipeline(config)

assert len(pipeline.dataset) == 483, (
    f"Esperaba 483 variantes, "
    f"obtuve {len(pipeline.dataset)}"
)

device = pipeline.device

print("=" * 90)
print("PIPELINE")
print("=" * 90)
print("device       =", device)
print("dataset_size =", len(pipeline.dataset))
print(
    "architecture =",
    config["model"]["architecture"]
)

# ------------------------------------------------------------
# 2. Cargar BEST checkpoint
# ------------------------------------------------------------

checkpoint = load_checkpoint(
    BEST,
    map_location="cpu",
)

print()
print("=" * 90)
print("BEST CHECKPOINT")
print("=" * 90)

print(
    "epoch_completed =",
    checkpoint["epoch_completed"]
)

print(
    "best_metric     =",
    checkpoint["best_metric"]
)

print(
    "architecture    =",
    checkpoint.get("architecture")
)

print(
    "seed            =",
    checkpoint.get("seed")
)

assert checkpoint["epoch_completed"] == 32
assert checkpoint.get("seed") == 11

pipeline.model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

pipeline.model.to(device)
pipeline.model.eval()

# ------------------------------------------------------------
# 3. Loader completo, sin shuffle
# ------------------------------------------------------------

loader = DataLoader(
    pipeline.dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_mut_wt_pairs,
)

# ------------------------------------------------------------
# 4. Extraer representación de una sola vista ORIGINAL
#    Sin augmentations.
# ------------------------------------------------------------

h_pair_parts = []
z_delta_parts = []
z_instance_parts = []
variant_ids = []

model = pipeline.model

one_view_model = (
    model.two_view_model.one_view_model
)

active_scales = (
    one_view_model
    .multiscale_pooling
    .enabled_scales
)

print()
print(
    "active_scales =",
    active_scales
)

with torch.no_grad():

    for batch_idx, raw_batch in enumerate(loader):

        batch = raw_batch.to(device)

        # -----------------------------------------------
        # Las mismas máscaras semánticas que Model A
        # usa en su forward productivo.
        # -----------------------------------------------

        mutation_mut = (
            batch.graph_mut.is_mutation.bool()
        )

        mutation_delta = mutation_mut[
            batch.mut_aligned_index
        ]

        mutation_wt = mutation_mut.new_zeros(
            batch.graph_wt.num_nodes
        )

        mutation_wt[
            batch.wt_aligned_index[
                mutation_delta
            ]
        ] = True

        masks = {
            "mutation_mask_MUT":
                mutation_mut,
            "mutation_mask_WT":
                mutation_wt,
            "mutation_mask_delta":
                mutation_delta,
        }

        if "local" in active_scales:

            masks.update(
                {
                    "local_mask_MUT":
                        indices_to_mask(
                            batch.local_mut_aligned_index,
                            row_count=(
                                batch.graph_mut.num_nodes
                            ),
                            name=(
                                "local_mut_aligned_index"
                            ),
                        ),

                    "local_mask_WT":
                        indices_to_mask(
                            batch.local_wt_aligned_index,
                            row_count=(
                                batch.graph_wt.num_nodes
                            ),
                            name=(
                                "local_wt_aligned_index"
                            ),
                        ),

                    "local_mask_delta":
                        aligned_selection_mask(
                            batch.mut_aligned_index,
                            batch.alignment_ptr,
                            batch.local_mut_aligned_index,
                            batch.local_alignment_ptr,
                            num_pairs=batch.batch_size,
                        ),
                }
            )

        # -----------------------------------------------
        # Una vista SIN augmentations
        # -----------------------------------------------

        output = one_view_model(
            graph_mut=batch.graph_mut,
            graph_wt=batch.graph_wt,
            mut_aligned_index=(
                batch.mut_aligned_index
            ),
            wt_aligned_index=(
                batch.wt_aligned_index
            ),
            aligned_pair_batch=(
                batch.aligned_pair_batch
            ),
            alignment_ptr=(
                batch.alignment_ptr
            ),
            num_pairs=batch.batch_size,
            variant_id=tuple(
                batch.variant_ids
            ),
            **masks,
        )

        # Las 483 variantes deberían producir
        # una representación válida.
        assert bool(
            output.pair_valid_mask.all()
        ), (
            f"pair_valid_mask falso "
            f"en batch {batch_idx}"
        )

        h_pair = output.h_pair_delta

        z_delta = output.z_delta_pair

        z_instance = (
            model.projection_pair_a(
                z_delta
            )
        )

        assert torch.isfinite(
            h_pair
        ).all()

        assert torch.isfinite(
            z_delta
        ).all()

        assert torch.isfinite(
            z_instance
        ).all()

        h_pair_parts.append(
            h_pair.detach().cpu().numpy()
        )

        z_delta_parts.append(
            z_delta.detach().cpu().numpy()
        )

        z_instance_parts.append(
            z_instance.detach().cpu().numpy()
        )

        variant_ids.extend(
            list(batch.variant_ids)
        )

# ------------------------------------------------------------
# 5. Concatenar
# ------------------------------------------------------------

h_pair_delta = np.concatenate(
    h_pair_parts,
    axis=0,
)

z_delta_pair = np.concatenate(
    z_delta_parts,
    axis=0,
)

z_instance_pair = np.concatenate(
    z_instance_parts,
    axis=0,
)

variant_ids = np.asarray(
    variant_ids,
    dtype=str,
)

# ------------------------------------------------------------
# 6. Contratos duros
# ------------------------------------------------------------

assert h_pair_delta.shape[0] == 483
assert z_delta_pair.shape[0] == 483
assert z_instance_pair.shape[0] == 483
assert variant_ids.shape[0] == 483

assert len(set(variant_ids.tolist())) == 483

assert np.isfinite(
    h_pair_delta
).all()

assert np.isfinite(
    z_delta_pair
).all()

assert np.isfinite(
    z_instance_pair
).all()

# No colapso exacto.
assert np.unique(
    h_pair_delta,
    axis=0
).shape[0] > 1

assert np.unique(
    z_delta_pair,
    axis=0
).shape[0] > 1

assert np.unique(
    z_instance_pair,
    axis=0
).shape[0] > 1

# ------------------------------------------------------------
# 7. Persistir
# ------------------------------------------------------------

np.savez_compressed(
    EXPORT_NPZ,
    h_pair_delta=h_pair_delta,
    z_delta_pair=z_delta_pair,
    z_instance_pair=z_instance_pair,
)

VARIANT_IDS_TXT.write_text(
    "\n".join(
        variant_ids.tolist()
    ) + "\n",
    encoding="utf-8",
)

print()
print("=" * 90)
print("A11 REPRESENTATION EXPORT")
print("=" * 90)

print(
    "h_pair_delta    =",
    h_pair_delta.shape
)

print(
    "z_delta_pair    =",
    z_delta_pair.shape
)

print(
    "z_instance_pair =",
    z_instance_pair.shape
)

print(
    "variant_ids     =",
    variant_ids.shape
)

print()
print(
    "unique h_pair_delta    =",
    np.unique(
        h_pair_delta,
        axis=0
    ).shape[0]
)

print(
    "unique z_delta_pair    =",
    np.unique(
        z_delta_pair,
        axis=0
    ).shape[0]
)

print(
    "unique z_instance_pair =",
    np.unique(
        z_instance_pair,
        axis=0
    ).shape[0]
)

print()
print("NPZ =", EXPORT_NPZ)
print(
    "VARIANT_IDS =",
    VARIANT_IDS_TXT
)

print()
print(
    "A11_PAIR_REPRESENTATIONS_EXPORT=PASS"
)

PIPELINE
device       = cuda
dataset_size = 483
architecture = model_a_nodal_multiscale_pair

BEST CHECKPOINT
epoch_completed = 32
best_metric     = 0.21073842965639555
architecture    = model_a_nodal_multiscale_pair
seed            = 11

active_scales = ('mutation', 'local', 'global')

A11 REPRESENTATION EXPORT
h_pair_delta    = (483, 1920)
z_delta_pair    = (483, 128)
z_instance_pair = (483, 64)
variant_ids     = (483,)

unique h_pair_delta    = 483
unique z_delta_pair    = 483
unique z_instance_pair = 483

NPZ = /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9/model_a_nodal_multiscale_pair/run_20260902T093129.973812Z-5c337698/exports/a9_pair_representations_best.npz
VARIANT_IDS = /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9/model_a_nodal_multiscale_pair/run_20260902T093129.973812Z-5c337698/exports/a9_pair_representations_variant_ids.txt

A11_PAIR_REPRESENTATIONS_EXPORT=PASS


In [ ]:
from pathlib import Path
import subprocess
import sys
import json

# ============================================================
# A9 · MODELO A · SEED 11
# ACCEPTANCE OFICIAL
# ============================================================

REPO_DIR = Path(
    "/content/model_a_workspace/repo"
)

A11 = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_a/runs/model_a_a9/"
    "model_a_nodal_multiscale_pair/"
    "run_20260902T093129.973812Z-5c337698"
)

CONFIG = (
    A11 / "config_resolved.yaml"
)

REPRESENTATIONS = (
    A11
    / "exports"
    / "a9_pair_representations_best.npz"
)

ACCEPTANCE_JSON = (
    A11 / "a9_acceptance.json"
)

# ------------------------------------------------------------
# Comprobaciones previas
# ------------------------------------------------------------

assert A11.is_dir(), A11
assert CONFIG.is_file(), CONFIG
assert REPRESENTATIONS.is_file(), REPRESENTATIONS

cmd = [
    sys.executable,
    "scripts/a9_acceptance.py",
    "--run-dir",
    str(A11),
    "--config",
    str(CONFIG),
    "--pair-representations",
    str(REPRESENTATIONS),
]

print("=" * 90)
print("A11 · A9 OFFICIAL ACCEPTANCE")
print("=" * 90)

print("COMMAND:")
print(" ".join(cmd))
print()

result = subprocess.run(
    cmd,
    cwd=str(REPO_DIR),
    text=True,
    capture_output=True,
)

print("RETURN_CODE =", result.returncode)

print()
print("STDOUT:")
print(result.stdout or "<empty>")

if result.stderr:
    print()
    print("STDERR:")
    print(result.stderr)

print()
print(
    "acceptance_json_exists =",
    ACCEPTANCE_JSON.exists()
)

acceptance = None

if ACCEPTANCE_JSON.exists():

    acceptance = json.loads(
        ACCEPTANCE_JSON.read_text(
            encoding="utf-8"
        )
    )

    print()
    print("=" * 90)
    print("A11 ACCEPTANCE SUMMARY")
    print("=" * 90)

    print(
        "status =",
        acceptance.get("status")
    )

    print(
        "run_seed =",
        acceptance.get("run_seed")
    )

    print(
        "architecture =",
        acceptance.get("architecture")
    )

    print(
        "automatic_rejection_checks =",
        acceptance.get(
            "automatic_rejection_checks"
        )
    )

    if acceptance.get("status") == "rejected":
        print(
            "reason =",
            acceptance.get("reason")
        )

    representation_metrics = (
        acceptance.get(
            "representation_metrics",
            {}
        )
    )

    if representation_metrics:

        print()
        print(
            "=" * 90
        )
        print(
            "REPRESENTATION METRICS"
        )
        print(
            "=" * 90
        )

        for name, metrics in (
            representation_metrics.items()
        ):

            print()
            print(name)

            for key, value in metrics.items():
                print(
                    f"  {key} = {value}"
                )

# ------------------------------------------------------------
# Veredicto
# ------------------------------------------------------------

print()
print("=" * 90)

if (
    result.returncode == 0
    and acceptance is not None
    and acceptance.get("status") == "accepted"
    and acceptance.get("run_seed") == 11
    and acceptance.get(
        "automatic_rejection_checks"
    ) == "PASS"
):

    print("A9_ACCEPTANCE_A11=ACCEPTED")
    print()
    print(
        "A11 queda formalmente cerrado."
    )
    print(
        "NEXT=A23"
    )

else:

    print(
        "A9_ACCEPTANCE_A11=NOT_ACCEPTED"
    )
    print(
        "NO iniciar seed 23."
    )

print("=" * 90)

A11 · A9 OFFICIAL ACCEPTANCE
COMMAND:
/usr/bin/python3 scripts/a9_acceptance.py --run-dir /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9/model_a_nodal_multiscale_pair/run_20260902T093129.973812Z-5c337698 --config /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9/model_a_nodal_multiscale_pair/run_20260902T093129.973812Z-5c337698/config_resolved.yaml --pair-representations /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9/model_a_nodal_multiscale_pair/run_20260902T093129.973812Z-5c337698/exports/a9_pair_representations_best.npz

RETURN_CODE = 0

STDOUT:
{
  "architecture": "model_a_nodal_multiscale_pair",
  "automatic_rejection_checks": "PASS",
  "representation_metrics": {
    "h_pair_delta": {
      "effective_rank": 3.8468099198320185,
      "exact_total_collapse": false,
      "exact_unique_rows": 483,
      "finite": true,
      "pc1_variance_fraction": 0.5853628011235709,
      "shape": [
        483,
        1920
      ]


# **A23 seed_training**

In [ ]:
from pathlib import Path
import sys
import os
import json
import subprocess

# ============================================================
# A23 · PREPARACIÓN Y PREFLIGHT
# NO INICIA ENTRENAMIENTO
# ============================================================

RUN_SEED = 23
SPLIT_SEED = 42

EXPECTED_COMMIT = (
    "3757bfdde52a9920611f3e7b8027434ce80d54bc"
)

ARCHITECTURE = "model_a_nodal_multiscale_pair"

REPO_DIR = Path(
    "/content/model_a_workspace/repo"
)

CONFIG_BASE = (
    REPO_DIR / "configs/model_a_a9.yaml"
)

MUTANTS_HDF5 = Path(
    "/content/model_a_workspace/staging/proc_483p.hdf5"
)

WT_HDF5 = Path(
    "/content/model_a_workspace/staging/wt_companion.hdf5"
)

A_OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_a/runs/model_a_a9"
)

B_OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_b/runs/model_b_a9"
)

ALLOWED_OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2"
)

PREFLIGHT_DIR = (
    A_OUTPUT_ROOT / "_preflight"
)

PREFLIGHT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PREFLIGHT_JSON = (
    PREFLIGHT_DIR
    / "model_a_seed23_a9_preflight.json"
)

RESOLVED_CONFIG_DIR = Path(
    "/content/model_a_workspace/resolved_configs"
)

RESOLVED_CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RUNTIME_CONFIG = (
    RESOLVED_CONFIG_DIR
    / "model_a_a9_seed23_runtime.yaml"
)

# ------------------------------------------------------------
# 0. Entorno
# ------------------------------------------------------------

os.chdir(REPO_DIR)

for candidate in (
    str(REPO_DIR),
    str(REPO_DIR / "src"),
):
    if candidate not in sys.path:
        sys.path.insert(0, candidate)

from gnn_siamese.config import save_config
from scripts.a9_preflight import (
    resolve_a9_runtime_configs,
    validate_a9_colab_preflight,
)

# ------------------------------------------------------------
# 1. Gate: A11 debe estar formalmente ACCEPTED
# ------------------------------------------------------------

A11 = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_a/runs/model_a_a9/"
    "model_a_nodal_multiscale_pair/"
    "run_20260902T093129.973812Z-5c337698"
)

A11_ACCEPTANCE = (
    A11 / "a9_acceptance.json"
)

assert A11_ACCEPTANCE.is_file(), (
    "No encuentro la acceptance persistida de A11."
)

a11_acceptance = json.loads(
    A11_ACCEPTANCE.read_text(
        encoding="utf-8"
    )
)

assert (
    a11_acceptance.get("status")
    == "accepted"
)

assert (
    a11_acceptance.get("run_seed")
    == 11
)

assert (
    a11_acceptance.get(
        "automatic_rejection_checks"
    )
    == "PASS"
)

print("=" * 90)
print("GATE A11")
print("=" * 90)
print("A11 = ACCEPTED")
print()

# ------------------------------------------------------------
# 2. Evitar lanzar A23 si ya hay otro train.py activo
# ------------------------------------------------------------

ps = subprocess.run(
    ["ps", "-eo", "pid,stat,cmd"],
    capture_output=True,
    text=True,
    check=True,
)

active_trainers = []

for line in ps.stdout.splitlines():

    if (
        "scripts/train.py" in line
        and "grep" not in line
    ):
        active_trainers.append(line)

print("=" * 90)
print("ACTIVE TRAINERS")
print("=" * 90)

if active_trainers:
    for line in active_trainers:
        print(line)

    raise RuntimeError(
        "Existe un scripts/train.py activo. "
        "NO preparar/lanzar otra seed todavía."
    )

print("Ningún scripts/train.py activo.")
print()

# ------------------------------------------------------------
# 3. Preflight A9 oficial para seed 23
# ------------------------------------------------------------

result = validate_a9_colab_preflight(
    CONFIG_BASE,
    architecture=ARCHITECTURE,
    run_seed=RUN_SEED,
    mutants_hdf5=MUTANTS_HDF5,
    wt_hdf5=WT_HDF5,
    output_root=A_OUTPUT_ROOT,
    peer_output_root=B_OUTPUT_ROOT,
    repo_root=REPO_DIR,
    expected_commit=EXPECTED_COMMIT,
    allowed_output_root=ALLOWED_OUTPUT_ROOT,
)

assert result["status"] == "PASS"
assert result["run_seed"] == 23
assert result["split_seed"] == 42
assert result["ab_intersection"] == 483
assert result["prospective_exists"] is False

# ------------------------------------------------------------
# 4. Persistir el preflight en Drive
# ------------------------------------------------------------

PREFLIGHT_JSON.write_text(
    json.dumps(
        result,
        indent=2,
        sort_keys=True,
        default=str,
    )
    + "\n",
    encoding="utf-8",
)

# ------------------------------------------------------------
# 5. Materializar configuración runtime A23
# ------------------------------------------------------------

config_a, config_b = (
    resolve_a9_runtime_configs(
        CONFIG_BASE,
        architecture=ARCHITECTURE,
        run_seed=RUN_SEED,
        mutants_hdf5=MUTANTS_HDF5,
        wt_hdf5=WT_HDF5,
        output_root=A_OUTPUT_ROOT,
        peer_output_root=B_OUTPUT_ROOT,
        repo_root=REPO_DIR,
    )
)

# Contratos críticos
assert config_a["project"]["seed"] == 23
assert config_a["split"]["seed"] == 42

assert (
    config_a["model"]["architecture"]
    == ARCHITECTURE
)

assert (
    config_a["training"]["epochs"]
    == 100
)

assert (
    config_a["training"]["batch_size"]
    == 4
)

assert (
    config_a["training"]["device"]
    == "cuda"
)

assert (
    config_a["training"]
    ["early_stopping"]
    ["patience"]
    == 15
)

# Guardar configuración runtime fuera del repo
save_config(
    config_a,
    RUNTIME_CONFIG,
)

# ------------------------------------------------------------
# 6. Resumen
# ------------------------------------------------------------

print("=" * 90)
print("A9 MODELO A · SEED 23")
print("=" * 90)

print("A9_PREFLIGHT = PASS")

print(
    "commit       =",
    result["git"],
)

print(
    "run_seed     =",
    result["run_seed"],
)

print(
    "split_seed   =",
    result["split_seed"],
)

print(
    "architecture =",
    config_a["model"]["architecture"],
)

print(
    "device       =",
    config_a["training"]["device"],
)

print(
    "epochs       =",
    config_a["training"]["epochs"],
)

print(
    "batch_size   =",
    config_a["training"]["batch_size"],
)

print(
    "scheduler    =",
    config_a["training"]["scheduler"],
)

print(
    "early_stop   =",
    config_a["training"]["early_stopping"],
)

print(
    "mutants_h5   =",
    config_a["paths"]["mutants_hdf5"],
)

print(
    "wt_h5        =",
    config_a["paths"]["wt_companion_hdf5"],
)

print(
    "output_root  =",
    config_a["outputs"]["root_dir"],
)

print()
print(
    "prospective_run =",
    result["prospective_run_dir"],
)

print()
print(
    "PREFLIGHT_JSON =",
    PREFLIGHT_JSON,
)

print(
    "RUNTIME_CONFIG =",
    RUNTIME_CONFIG,
)

print()
print("READY_FOR_A23_TRAINING=YES")

GATE A11
A11 = ACCEPTED

ACTIVE TRAINERS
Ningún scripts/train.py activo.

A9 MODELO A · SEED 23
A9_PREFLIGHT = PASS
commit       = {'commit': '3757bfdde52a9920611f3e7b8027434ce80d54bc', 'expected_commit': '3757bfdde52a9920611f3e7b8027434ce80d54bc', 'working_tree': 'clean'}
run_seed     = 23
split_seed   = 42
architecture = model_a_nodal_multiscale_pair
device       = cuda
epochs       = 100
batch_size   = 4
scheduler    = cosine
early_stop   = {'enabled': True, 'monitor': 'validation_loss', 'mode': 'min', 'patience': 15, 'min_delta': 0.0}
mutants_h5   = /content/model_a_workspace/staging/proc_483p.hdf5
wt_h5        = /content/model_a_workspace/staging/wt_companion.hdf5
output_root  = /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9

prospective_run = /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9/model_a_nodal_multiscale_pair/run_20260903T080709.565419Z-2915f3ea

PREFLIGHT_JSON = /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a

In [ ]:
from pathlib import Path
import subprocess
import sys
import os
import json
import time
from datetime import datetime, timezone

# ============================================================
# A23 · MODELO A · SEED 23
# LANZAMIENTO FRESH PROTEGIDO
#
# ESTA CELDA SÍ INICIA ENTRENAMIENTO
# ============================================================

RUN_SEED = 23
ARCHITECTURE = "model_a_nodal_multiscale_pair"

REPO_DIR = Path(
    "/content/model_a_workspace/repo"
)

RUNTIME_CONFIG = Path(
    "/content/model_a_workspace/"
    "resolved_configs/model_a_a9_seed23_runtime.yaml"
)

A_OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_a/runs/model_a_a9"
)

ARCH_ROOT = (
    A_OUTPUT_ROOT / ARCHITECTURE
)

LAUNCHER_DIR = (
    A_OUTPUT_ROOT / "_launcher_logs"
)

LAUNCHER_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LOG = (
    LAUNCHER_DIR
    / "model_a_seed23_training.log"
)

LAUNCH_RECORD = (
    LAUNCHER_DIR
    / "model_a_seed23_launcher.json"
)

PID_FILE = Path(
    "/content/model_a_workspace/a23.pid"
)

RUN_DIR_FILE = Path(
    "/content/model_a_workspace/a23_run_dir.txt"
)

# ============================================================
# 1. Contratos previos
# ============================================================

assert REPO_DIR.is_dir(), REPO_DIR
assert RUNTIME_CONFIG.is_file(), RUNTIME_CONFIG

os.chdir(REPO_DIR)

for candidate in (
    str(REPO_DIR),
    str(REPO_DIR / "src"),
):
    if candidate not in sys.path:
        sys.path.insert(0, candidate)

from gnn_siamese.config import load_config

config = load_config(RUNTIME_CONFIG)

assert config["project"]["seed"] == 23
assert config["split"]["seed"] == 42

assert (
    config["model"]["architecture"]
    == ARCHITECTURE
)

assert config["training"]["epochs"] == 100
assert config["training"]["batch_size"] == 4
assert config["training"]["device"] == "cuda"

print("=" * 90)
print("A23 · PRE-LAUNCH GUARDS")
print("=" * 90)

print("run_seed     =", config["project"]["seed"])
print("split_seed   =", config["split"]["seed"])
print("architecture =", config["model"]["architecture"])
print("device       =", config["training"]["device"])

# ============================================================
# 2. No debe haber ningún trainer activo
# ============================================================

ps = subprocess.run(
    ["ps", "-eo", "pid,stat,cmd"],
    capture_output=True,
    text=True,
    check=True,
)

active_trainers = [
    line
    for line in ps.stdout.splitlines()
    if (
        "scripts/train.py" in line
        and "grep" not in line
    )
]

print()
print("active_trainers =", len(active_trainers))

if active_trainers:

    for line in active_trainers:
        print(line)

    raise RuntimeError(
        "Ya existe un scripts/train.py activo. "
        "NO se inicia A23."
    )

# ============================================================
# 3. Buscar cualquier seed 23 previamente creada
#    Esto evita duplicados incluso tras reconexiones.
# ============================================================

existing_seed23 = []

if ARCH_ROOT.exists():

    for run_dir in sorted(
        ARCH_ROOT.glob("run_*")
    ):

        manifest_path = (
            run_dir / "run_manifest.json"
        )

        if not manifest_path.is_file():
            continue

        try:
            manifest = json.loads(
                manifest_path.read_text(
                    encoding="utf-8"
                )
            )
        except Exception:
            continue

        seed = (
            manifest
            .get("configuration", {})
            .get("seed")
        )

        if seed == 23:

            existing_seed23.append(
                {
                    "run_dir": run_dir,
                    "status":
                        manifest.get("status"),
                    "stage":
                        manifest.get(
                            "lifecycle", {}
                        ).get("stage"),
                    "epochs":
                        manifest.get(
                            "training", {}
                        ).get(
                            "epochs_completed"
                        ),
                }
            )

print(
    "existing_seed23_runs =",
    len(existing_seed23)
)

if existing_seed23:

    print()

    for item in existing_seed23:

        print(item["run_dir"])
        print(
            "  status =",
            item["status"]
        )
        print(
            "  stage  =",
            item["stage"]
        )
        print(
            "  epochs =",
            item["epochs"]
        )

    raise RuntimeError(
        "Ya existe al menos un run seed 23. "
        "NO crear otro. Hay que inspeccionarlo."
    )

# Snapshot antes del lanzamiento
before_runs = set(
    ARCH_ROOT.glob("run_*")
) if ARCH_ROOT.exists() else set()

# ============================================================
# 4. Lanzamiento
# ============================================================

cmd = [
    sys.executable,
    "-u",
    "scripts/train.py",
    "--config",
    str(RUNTIME_CONFIG),
    "--device",
    "cuda",
]

print()
print("=" * 90)
print("A23 · LAUNCH")
print("=" * 90)

print("COMMAND:")
print(" ".join(cmd))

# Vaciar solo el log de seed 23.
# No toca ningún run.
log_handle = open(
    LOG,
    "w",
    encoding="utf-8",
    buffering=1,
)

process = subprocess.Popen(
    cmd,
    cwd=str(REPO_DIR),
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

PID = process.pid

PID_FILE.write_text(
    str(PID) + "\n",
    encoding="utf-8",
)

print()
print("PID =", PID)
print("LOG =", LOG)

# ============================================================
# 5. Esperar a que bootstrap cree el nuevo run
# ============================================================

new_run = None

for _ in range(120):

    current_runs = set(
        ARCH_ROOT.glob("run_*")
    ) if ARCH_ROOT.exists() else set()

    created = sorted(
        current_runs - before_runs
    )

    if len(created) == 1:

        candidate = created[0]

        manifest_path = (
            candidate / "run_manifest.json"
        )

        if manifest_path.is_file():

            try:
                manifest = json.loads(
                    manifest_path.read_text(
                        encoding="utf-8"
                    )
                )

                seed = (
                    manifest
                    .get("configuration", {})
                    .get("seed")
                )

                if seed == 23:
                    new_run = candidate
                    break

            except Exception:
                pass

    elif len(created) > 1:

        raise RuntimeError(
            "Aparecieron múltiples run_* nuevos "
            "después del lanzamiento. "
            "No continuar sin inspección."
        )

    # Si el proceso murió durante startup
    if process.poll() is not None:
        break

    time.sleep(1)

# Ya podemos cerrar nuestro descriptor local.
# El proceso hijo mantiene el suyo.
log_handle.close()

# ============================================================
# 6. Verificación inmediata
# ============================================================

return_code = process.poll()

if new_run is None:

    print()
    print("=" * 90)
    print("A23 STARTUP FAILURE / UNRESOLVED")
    print("=" * 90)

    print(
        "process_return_code =",
        return_code
    )

    if LOG.exists():

        print()
        print("LOG:")

        lines = LOG.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()

        for line in lines[-50:]:
            print(line)

    raise RuntimeError(
        "No se pudo identificar inequívocamente "
        "el run A23. NO relanzar."
    )

RUN_DIR_FILE.write_text(
    str(new_run) + "\n",
    encoding="utf-8",
)

manifest_path = (
    new_run / "run_manifest.json"
)

manifest = json.loads(
    manifest_path.read_text(
        encoding="utf-8"
    )
)

assert (
    manifest
    .get("configuration", {})
    .get("seed")
    == 23
)

assert (
    manifest.get("architecture")
    == ARCHITECTURE
)

status = manifest.get("status")

assert status == "running", (
    f"Estado inicial inesperado: {status}"
)

# Persistir también referencia en Drive
# para recuperarla tras una desconexión.
launch_record = {
    "architecture": ARCHITECTURE,
    "run_seed": 23,
    "split_seed": 42,
    "pid": PID,
    "run_dir": str(new_run),
    "runtime_config": str(RUNTIME_CONFIG),
    "log": str(LOG),
    "launched_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

LAUNCH_RECORD.write_text(
    json.dumps(
        launch_record,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

# ============================================================
# 7. Resultado
# ============================================================

print()
print("=" * 90)
print("A23 STARTED")
print("=" * 90)

print("PID      =", PID)
print("RUN_DIR  =", new_run)

print(
    "status   =",
    manifest.get("status")
)

print(
    "stage    =",
    manifest.get(
        "lifecycle", {}
    ).get("stage")
)

print(
    "seed     =",
    manifest.get(
        "configuration", {}
    ).get("seed")
)

print("LOG      =", LOG)

print(
    "LAUNCH_RECORD =",
    LAUNCH_RECORD
)

print()
print("A23_BACKGROUND_TRAINING=RUNNING")

A23 · PRE-LAUNCH GUARDS
run_seed     = 23
split_seed   = 42
architecture = model_a_nodal_multiscale_pair
device       = cuda

active_trainers = 1
   2021 Rsl  /usr/bin/python3 -u scripts/train.py --config /content/model_a_workspace/resolved_configs/model_a_a9_seed23_runtime.yaml --device cuda


RuntimeError: Ya existe un scripts/train.py activo. NO se inicia A23.

In [ ]:
from pathlib import Path
import json
import math
import subprocess

# ============================================================
# A23 · EARLY HEALTH CHECK
# SOLO LECTURA — NO LANZA ENTRENAMIENTO
# ============================================================

ARCHITECTURE = "model_a_nodal_multiscale_pair"

LAUNCH_RECORD = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_a/runs/model_a_a9/_launcher_logs/"
    "model_a_seed23_launcher.json"
)

assert LAUNCH_RECORD.is_file(), LAUNCH_RECORD

launch = json.loads(
    LAUNCH_RECORD.read_text(encoding="utf-8")
)

assert launch["run_seed"] == 23
assert launch["architecture"] == ARCHITECTURE

RUN_DIR = Path(launch["run_dir"])
LOG = Path(launch["log"])

MANIFEST = RUN_DIR / "run_manifest.json"
METRICS = RUN_DIR / "metrics.jsonl"
BEST = RUN_DIR / "checkpoints/best.pt"
LAST = RUN_DIR / "checkpoints/last.pt"

print("=" * 90)
print("A23 · EARLY HEALTH CHECK")
print("=" * 90)

print("RUN_DIR =", RUN_DIR)

# ------------------------------------------------------------
# 1. Proceso activo
# ------------------------------------------------------------

ps = subprocess.run(
    ["ps", "-eo", "pid,stat,cmd"],
    capture_output=True,
    text=True,
    check=True,
)

trainers = [
    line
    for line in ps.stdout.splitlines()
    if "scripts/train.py" in line
    and "grep" not in line
]

print()
print("ACTIVE TRAINERS =", len(trainers))

for line in trainers:
    print(line)

# ------------------------------------------------------------
# 2. Manifest
# ------------------------------------------------------------

assert MANIFEST.is_file(), MANIFEST

manifest = json.loads(
    MANIFEST.read_text(encoding="utf-8")
)

status = manifest.get("status")
stage = manifest.get("lifecycle", {}).get("stage")

seed = (
    manifest
    .get("configuration", {})
    .get("seed")
)

split_seed = (
    manifest
    .get("configuration", {})
    .get("seed_bundle", {})
    .get("split")
)

training = manifest.get("training", {})

print()
print("=" * 90)
print("MANIFEST")
print("=" * 90)

print("status           =", status)
print("stage            =", stage)
print("architecture     =", manifest.get("architecture"))
print("run_seed         =", seed)
print("split_seed       =", split_seed)
print("epochs_completed =", training.get("epochs_completed"))
print("global_step      =", training.get("global_step"))
print("batch_size       =", training.get("batch_size"))

assert manifest.get("architecture") == ARCHITECTURE
assert seed == 23
assert split_seed == 42
assert training.get("batch_size") == 4

# ------------------------------------------------------------
# 3. Métricas
# ------------------------------------------------------------

rows = []

if METRICS.is_file():

    for line in METRICS.read_text(
        encoding="utf-8"
    ).splitlines():

        if line.strip():
            rows.append(json.loads(line))

print()
print("=" * 90)
print("METRICS")
print("=" * 90)

print("metric_records =", len(rows))

nonfinite = []

for row in rows:

    epoch = row["epoch"]

    train_loss = float(
        row["train"]["mean_loss"]
    )

    val_loss = float(
        row["validation"]["mean_loss"]
    )

    if not (
        math.isfinite(train_loss)
        and math.isfinite(val_loss)
    ):
        nonfinite.append(epoch)

print("nonfinite_epochs =", nonfinite)

if rows:

    print()
    print("ÉPOCAS DISPONIBLES")

    for row in rows[:3]:

        print(
            f"epoch={row['epoch']} "
            f"step={row['global_step']} "
            f"train={row['train']['mean_loss']:.6f} "
            f"val={row['validation']['mean_loss']:.6f}"
        )

    if len(rows) > 3:

        print("...")
        row = rows[-1]

        print(
            f"latest: epoch={row['epoch']} "
            f"step={row['global_step']} "
            f"train={row['train']['mean_loss']:.6f} "
            f"val={row['validation']['mean_loss']:.6f}"
        )

# ------------------------------------------------------------
# 4. Checkpoints
# ------------------------------------------------------------

print()
print("=" * 90)
print("CHECKPOINTS")
print("=" * 90)

print("best.pt =", BEST.exists())
print("last.pt =", LAST.exists())

# ------------------------------------------------------------
# 5. Buscar errores reales en el log
# ------------------------------------------------------------

log_text = ""

if LOG.is_file():
    log_text = LOG.read_text(
        encoding="utf-8",
        errors="replace",
    )

serious_patterns = [
    "Traceback (most recent call last)",
    "CUDA out of memory",
    "RuntimeError:",
    "error=",
]

found_errors = [
    pattern
    for pattern in serious_patterns
    if pattern in log_text
]

print()
print("=" * 90)
print("LOG")
print("=" * 90)

print("log_size_lines =", len(log_text.splitlines()))
print("serious_patterns =", found_errors)

if found_errors:

    print()
    print("ÚLTIMAS 30 LÍNEAS DEL LOG")

    for line in log_text.splitlines()[-30:]:
        print(line)

# ------------------------------------------------------------
# 6. VEREDICTO
# ------------------------------------------------------------

print()
print("=" * 90)

if status == "completed":

    print("A23_TRAINING=COMPLETED")
    print("NEXT=POST_TRAINING_AUDIT")

elif status in {"failed", "interrupted"}:

    print(f"A23_TRAINING={status.upper()}")
    print("NO RELANZAR.")
    print("Enviar esta salida para diagnóstico.")

elif len(trainers) == 0 and status == "running":

    print("A23_HEALTH=UNRESOLVED")
    print(
        "Manifest sigue en running pero no hay train.py activo."
    )
    print("NO RELANZAR.")
    print("Enviar esta salida.")

elif nonfinite:

    print("A23_EARLY_HEALTH=FAIL")
    print("Se detectaron pérdidas no finitas.")
    print("NO continuar ni relanzar.")

elif found_errors:

    print("A23_EARLY_HEALTH=FAIL")
    print("Se detectó un error serio en el log.")
    print("NO relanzar.")

elif len(rows) < 3:

    print("A23_EARLY_HEALTH=WAIT")
    print(
        f"Solo hay {len(rows)} épocas completas. "
        "Deja entrenar y vuelve a ejecutar ESTA MISMA celda."
    )

elif BEST.exists() and LAST.exists():

    print("A23_EARLY_HEALTH=PASS")
    print("Primeras >=3 épocas finitas.")
    print("Checkpoints presentes.")
    print("Trainer activo.")
    print()
    print("CONTINUE_A23_UNINTERRUPTED=YES")

else:

    print("A23_EARLY_HEALTH=WAIT")
    print(
        "Las métricas existen pero aún faltan "
        "checkpoints esperados."
    )

print("=" * 90)

A23 · EARLY HEALTH CHECK
RUN_DIR = /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9/model_a_nodal_multiscale_pair/run_20260903T081000.855970Z-4995095d

ACTIVE TRAINERS = 1
   2021 Rsl  /usr/bin/python3 -u scripts/train.py --config /content/model_a_workspace/resolved_configs/model_a_a9_seed23_runtime.yaml --device cuda

MANIFEST
status           = running
stage            = training
architecture     = model_a_nodal_multiscale_pair
run_seed         = 23
split_seed       = 42
epochs_completed = 25
global_step      = 2150
batch_size       = 4

METRICS
metric_records = 25
nonfinite_epochs = []

ÉPOCAS DISPONIBLES
epoch=1 step=86 train=1.024302 val=0.778939
epoch=2 step=172 train=0.706546 val=0.792918
epoch=3 step=258 train=0.606166 val=0.602291
...
latest: epoch=25 step=2150 train=0.289898 val=0.278387

CHECKPOINTS
best.pt = True
last.pt = True

LOG
log_size_lines = 0
serious_patterns = []

A23_EARLY_HEALTH=PASS
Primeras >=3 épocas finitas.
Checkpoints presentes.
Trainer

# **Celda única — entrenamientos en cola A23 → A37 → A41 → A53**

In [ ]:
from pathlib import Path
import subprocess
import sys
import os
import json
import time
import textwrap
from datetime import datetime, timezone

# =============================================================================
# MODELO A · A9 · COLA MULTISEED SECUENCIAL PROTEGIDA
#
# A23 ya está corriendo:
#
#   A23 -> esperar finalización
#       -> A37 fresh
#       -> A41 fresh
#       -> A53 fresh
#
# SOLO UN ENTRENAMIENTO A LA VEZ.
#
# NO hace todavía:
#   - export de representaciones
#   - acceptance oficial
#
# Eso se hará después, seed por seed, usando cada best.pt.
# =============================================================================

REPO_DIR = Path(
    "/content/model_a_workspace/repo"
)

WORKSPACE = Path(
    "/content/model_a_workspace"
)

A9_ROOT = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_a/runs/model_a_a9"
)

QUEUE_DIR = (
    A9_ROOT / "_queue"
)

QUEUE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SUPERVISOR_SCRIPT = (
    WORKSPACE
    / "a9_model_a_multiseed_queue_supervisor.py"
)

QUEUE_LOG = (
    QUEUE_DIR
    / "model_a_a9_multiseed_queue.log"
)

QUEUE_STATE = (
    QUEUE_DIR
    / "model_a_a9_multiseed_queue_state.json"
)

QUEUE_LAUNCHER = (
    QUEUE_DIR
    / "model_a_a9_multiseed_queue_launcher.json"
)

LOCAL_PID = (
    WORKSPACE
    / "a9_model_a_multiseed_queue.pid"
)

assert REPO_DIR.is_dir(), REPO_DIR

# =============================================================================
# 0. GUARD: no crear un segundo supervisor
# =============================================================================

ps = subprocess.run(
    ["ps", "-eo", "pid,stat,cmd"],
    capture_output=True,
    text=True,
    check=True,
)

existing_supervisors = []

for line in ps.stdout.splitlines():

    if (
        "a9_model_a_multiseed_queue_supervisor.py"
        in line
        and "grep" not in line
    ):
        existing_supervisors.append(line)

print("=" * 100)
print("MODELO A · A9 MULTISEED QUEUE")
print("=" * 100)

if existing_supervisors:

    print("QUEUE_ALREADY_RUNNING")
    print()

    for line in existing_supervisors:
        print(line)

    print()
    print("No se crea un segundo supervisor.")
    print("QUEUE_LOG   =", QUEUE_LOG)
    print("QUEUE_STATE =", QUEUE_STATE)

    raise SystemExit

# =============================================================================
# 1. CREAR SUPERVISOR
# =============================================================================

supervisor_code = r'''
from __future__ import annotations

from pathlib import Path
import subprocess
import sys
import os
import json
import math
import time
import signal
import traceback
from datetime import datetime, timezone


# =============================================================================
# CONSTANTES
# =============================================================================

EXPECTED_COMMIT = (
    "3757bfdde52a9920611f3e7b8027434ce80d54bc"
)

ARCHITECTURE = (
    "model_a_nodal_multiscale_pair"
)

SPLIT_SEED = 42

SEEDS = [
    23,
    37,
    41,
    53,
]

MONITOR_INTERVAL_SECONDS = 30

EXPECTED_BATCHES_PER_EPOCH = 86


REPO_DIR = Path(
    "/content/model_a_workspace/repo"
)

WORKSPACE = Path(
    "/content/model_a_workspace"
)

CONFIG_BASE = (
    REPO_DIR
    / "configs"
    / "model_a_a9.yaml"
)

MUTANTS_HDF5 = Path(
    "/content/model_a_workspace/"
    "staging/proc_483p.hdf5"
)

WT_HDF5 = Path(
    "/content/model_a_workspace/"
    "staging/wt_companion.hdf5"
)

A_OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_a/runs/model_a_a9"
)

B_OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_b/runs/model_b_a9"
)

ALLOWED_OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2"
)

ARCH_ROOT = (
    A_OUTPUT_ROOT
    / ARCHITECTURE
)

PREFLIGHT_DIR = (
    A_OUTPUT_ROOT
    / "_preflight"
)

LAUNCHER_DIR = (
    A_OUTPUT_ROOT
    / "_launcher_logs"
)

QUEUE_DIR = (
    A_OUTPUT_ROOT
    / "_queue"
)

QUEUE_STATE = (
    QUEUE_DIR
    / "model_a_a9_multiseed_queue_state.json"
)

for directory in (
    PREFLIGHT_DIR,
    LAUNCHER_DIR,
    QUEUE_DIR,
    WORKSPACE / "resolved_configs",
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# =============================================================================
# PYTHON / REPO
# =============================================================================

os.chdir(REPO_DIR)

for candidate in (
    str(REPO_DIR),
    str(REPO_DIR / "src"),
):
    if candidate not in sys.path:
        sys.path.insert(
            0,
            candidate,
        )

from gnn_siamese.config import save_config

from scripts.a9_preflight import (
    resolve_a9_runtime_configs,
    validate_a9_colab_preflight,
)


# =============================================================================
# HELPERS
# =============================================================================

class QueueError(RuntimeError):
    pass


def utcnow():
    return datetime.now(
        timezone.utc
    ).isoformat()


def read_json(path: Path):

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def atomic_json(
    path: Path,
    payload: dict,
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_name(
        path.name + ".tmp"
    )

    tmp.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
            default=str,
        )
        + "\n",
        encoding="utf-8",
    )

    os.replace(
        tmp,
        path,
    )


state = {
    "status": "initializing",
    "architecture": ARCHITECTURE,
    "split_seed": SPLIT_SEED,
    "queue_seeds": SEEDS,
    "completed_seeds": [],
    "current_seed": None,
    "current_run_dir": None,
    "started_at_utc": utcnow(),
    "updated_at_utc": utcnow(),
}


def update_state(**kwargs):

    state.update(kwargs)

    state["updated_at_utc"] = (
        utcnow()
    )

    atomic_json(
        QUEUE_STATE,
        state,
    )


def get_active_trainers():

    result = subprocess.run(
        [
            "ps",
            "-eo",
            "pid,stat,cmd",
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    trainers = []

    for line in (
        result.stdout.splitlines()
    ):

        if (
            "scripts/train.py"
            not in line
        ):
            continue

        if "grep" in line:
            continue

        parts = line.strip().split(
            maxsplit=2
        )

        if len(parts) < 3:
            continue

        trainers.append(
            {
                "pid": int(parts[0]),
                "stat": parts[1],
                "cmd": parts[2],
            }
        )

    return trainers


def trainer_matches_seed(
    trainer: dict,
    seed: int,
):

    token = (
        f"seed{seed}_runtime.yaml"
    )

    return (
        token
        in trainer["cmd"]
    )


def find_seed_runs(
    seed: int,
):

    matches = []

    if not ARCH_ROOT.exists():
        return matches

    for run_dir in sorted(
        ARCH_ROOT.glob("run_*")
    ):

        manifest_path = (
            run_dir
            / "run_manifest.json"
        )

        if not manifest_path.is_file():
            continue

        try:

            manifest = read_json(
                manifest_path
            )

        except Exception:

            continue

        manifest_seed = (
            manifest
            .get(
                "configuration",
                {},
            )
            .get("seed")
        )

        if manifest_seed == seed:

            matches.append(
                run_dir
            )

    return matches


def read_metrics(
    run_dir: Path,
):

    metrics_path = (
        run_dir
        / "metrics.jsonl"
    )

    if not metrics_path.is_file():
        return []

    rows = []

    for line in (
        metrics_path
        .read_text(
            encoding="utf-8"
        )
        .splitlines()
    ):

        if line.strip():

            rows.append(
                json.loads(line)
            )

    return rows


def nonfinite_epochs(rows):

    bad = []

    for row in rows:

        epoch = int(
            row["epoch"]
        )

        train_loss = float(
            row["train"][
                "mean_loss"
            ]
        )

        val_loss = float(
            row["validation"][
                "mean_loss"
            ]
        )

        if not (
            math.isfinite(
                train_loss
            )
            and
            math.isfinite(
                val_loss
            )
        ):

            bad.append(epoch)

    return bad


def launcher_path(
    seed: int,
):

    return (
        LAUNCHER_DIR
        / f"model_a_seed{seed}_launcher.json"
    )


def training_log_path(
    seed: int,
):

    return (
        LAUNCHER_DIR
        / f"model_a_seed{seed}_training.log"
    )


def runtime_config_path(
    seed: int,
):

    return (
        WORKSPACE
        / "resolved_configs"
        / f"model_a_a9_seed{seed}_runtime.yaml"
    )


def serious_log_patterns(
    log_path: Path,
):

    if not log_path.is_file():
        return []

    text = log_path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    low = text.lower()

    patterns = {
        "traceback":
            "traceback (most recent call last)",

        "cuda_oom":
            "cuda out of memory",

        "runtime_error":
            "error=runtimeerror",

        "value_error":
            "error=valueerror",
    }

    found = []

    for name, token in (
        patterns.items()
    ):

        if token in low:
            found.append(name)

    return found


def interrupt_seed_if_running(
    seed: int,
):

    trainers = (
        get_active_trainers()
    )

    matching = [
        trainer
        for trainer in trainers
        if trainer_matches_seed(
            trainer,
            seed,
        )
    ]

    if len(matching) == 1:

        pid = matching[0]["pid"]

        print(
            f"[seed {seed}] "
            f"Enviando SIGINT protector "
            f"a PID {pid}",
            flush=True,
        )

        try:

            os.kill(
                pid,
                signal.SIGINT,
            )

        except ProcessLookupError:

            pass


def wait_no_trainers(
    timeout_seconds=120,
):

    deadline = (
        time.time()
        + timeout_seconds
    )

    while time.time() < deadline:

        trainers = (
            get_active_trainers()
        )

        if not trainers:
            return

        time.sleep(2)

    raise QueueError(
        "Sigue existiendo al menos "
        "un scripts/train.py activo "
        "cuando debería haber terminado."
    )


# =============================================================================
# A11 GATE
# =============================================================================

def validate_a11_gate():

    accepted = []

    for run_dir in (
        find_seed_runs(11)
    ):

        acceptance_path = (
            run_dir
            / "a9_acceptance.json"
        )

        if not (
            acceptance_path.is_file()
        ):
            continue

        try:

            payload = read_json(
                acceptance_path
            )

        except Exception:

            continue

        if (
            payload.get("status")
            == "accepted"
            and
            payload.get("run_seed")
            == 11
            and
            payload.get(
                "automatic_rejection_checks"
            )
            == "PASS"
        ):

            accepted.append(
                run_dir
            )

    if len(accepted) != 1:

        raise QueueError(
            "Debe existir exactamente "
            "un A11 oficialmente ACCEPTED; "
            f"encontrados={len(accepted)}"
        )

    print(
        "A11_GATE=ACCEPTED",
        accepted[0],
        flush=True,
    )

    return accepted[0]


# =============================================================================
# BASIC POST-TRAINING GATE
#
# No sustituye A9 acceptance.
# Solo permite decidir si es seguro lanzar la siguiente seed.
# =============================================================================

def completed_gate(
    run_dir: Path,
    seed: int,
):

    manifest_path = (
        run_dir
        / "run_manifest.json"
    )

    if not manifest_path.is_file():

        raise QueueError(
            f"Seed {seed}: "
            "falta run_manifest.json"
        )

    manifest = read_json(
        manifest_path
    )

    status = manifest.get(
        "status"
    )

    if status != "completed":

        raise QueueError(
            f"Seed {seed}: "
            f"status={status!r}, "
            "no completed."
        )

    if (
        manifest.get(
            "architecture"
        )
        != ARCHITECTURE
    ):

        raise QueueError(
            f"Seed {seed}: "
            "architecture mismatch."
        )

    manifest_seed = (
        manifest
        .get(
            "configuration",
            {},
        )
        .get("seed")
    )

    if manifest_seed != seed:

        raise QueueError(
            f"Seed {seed}: "
            f"manifest seed="
            f"{manifest_seed}"
        )

    manifest_split_seed = (
        manifest
        .get(
            "configuration",
            {},
        )
        .get(
            "seed_bundle",
            {},
        )
        .get("split")
    )

    if (
        manifest_split_seed
        != SPLIT_SEED
    ):

        raise QueueError(
            f"Seed {seed}: "
            "split seed mismatch: "
            f"{manifest_split_seed}"
        )

    training = manifest.get(
        "training",
        {},
    )

    if (
        training.get("batch_size")
        != 4
    ):

        raise QueueError(
            f"Seed {seed}: "
            "batch_size != 4"
        )

    required = [
        run_dir
        / "metrics.jsonl",

        run_dir
        / "gradient_audit.json",

        run_dir
        / "config_resolved.yaml",

        run_dir
        / "split.json",

        run_dir
        / "checkpoints"
        / "best.pt",

        run_dir
        / "checkpoints"
        / "last.pt",
    ]

    missing = [
        str(path)
        for path in required
        if (
            not path.is_file()
            or
            path.stat().st_size == 0
        )
    ]

    if missing:

        raise QueueError(
            f"Seed {seed}: "
            f"artefactos ausentes: "
            f"{missing}"
        )

    rows = read_metrics(
        run_dir
    )

    if not rows:

        raise QueueError(
            f"Seed {seed}: "
            "metrics.jsonl vacío."
        )

    bad = nonfinite_epochs(
        rows
    )

    if bad:

        raise QueueError(
            f"Seed {seed}: "
            f"NaN/Inf en épocas "
            f"{bad}"
        )

    epochs_completed = int(
        training.get(
            "epochs_completed",
            0,
        )
    )

    if (
        epochs_completed
        != len(rows)
    ):

        raise QueueError(
            f"Seed {seed}: "
            "epochs_completed != "
            "metric_records: "
            f"{epochs_completed} "
            f"vs {len(rows)}"
        )

    global_step = int(
        training.get(
            "global_step",
            0,
        )
    )

    expected_steps = (
        epochs_completed
        * EXPECTED_BATCHES_PER_EPOCH
    )

    if (
        global_step
        != expected_steps
    ):

        raise QueueError(
            f"Seed {seed}: "
            f"global_step={global_step}, "
            f"esperado={expected_steps}"
        )

    log_errors = (
        serious_log_patterns(
            training_log_path(
                seed
            )
        )
    )

    if log_errors:

        raise QueueError(
            f"Seed {seed}: "
            "errores serios en log: "
            f"{log_errors}"
        )

    summary = {
        "seed": seed,
        "run_dir": str(
            run_dir
        ),
        "epochs_completed":
            epochs_completed,
        "global_step":
            global_step,
        "stopped_early":
            training.get(
                "stopped_early"
            ),
        "stop_reason":
            training.get(
                "stop_reason"
            ),
        "best_metric":
            training.get(
                "best_metric"
            ),
        "metric_records":
            len(rows),
        "basic_gate":
            "PASS",
    }

    print(
        "=" * 90,
        flush=True,
    )

    print(
        f"SEED {seed} "
        f"BASIC_POST_TRAINING_GATE=PASS",
        flush=True,
    )

    print(
        json.dumps(
            summary,
            indent=2,
            sort_keys=True,
        ),
        flush=True,
    )

    print(
        "=" * 90,
        flush=True,
    )

    return summary


# =============================================================================
# PREFLIGHT + RUNTIME CONFIG
# =============================================================================

def prepare_fresh_seed(
    seed: int,
):

    trainers = (
        get_active_trainers()
    )

    if trainers:

        raise QueueError(
            f"Seed {seed}: "
            "hay train.py activo antes "
            "del preflight."
        )

    existing = (
        find_seed_runs(
            seed
        )
    )

    if existing:

        raise QueueError(
            f"Seed {seed}: "
            "ya existe un run; "
            "fresh bloqueado."
        )

    print(
        f"[seed {seed}] "
        "Ejecutando A9 preflight...",
        flush=True,
    )

    result = (
        validate_a9_colab_preflight(
            CONFIG_BASE,
            architecture=
                ARCHITECTURE,
            run_seed=
                seed,
            mutants_hdf5=
                MUTANTS_HDF5,
            wt_hdf5=
                WT_HDF5,
            output_root=
                A_OUTPUT_ROOT,
            peer_output_root=
                B_OUTPUT_ROOT,
            repo_root=
                REPO_DIR,
            expected_commit=
                EXPECTED_COMMIT,
            allowed_output_root=
                ALLOWED_OUTPUT_ROOT,
        )
    )

    if (
        result.get("status")
        != "PASS"
    ):

        raise QueueError(
            f"Seed {seed}: "
            "preflight != PASS"
        )

    if (
        result.get("run_seed")
        != seed
    ):

        raise QueueError(
            f"Seed {seed}: "
            "preflight run seed mismatch."
        )

    if (
        result.get("split_seed")
        != SPLIT_SEED
    ):

        raise QueueError(
            f"Seed {seed}: "
            "preflight split seed mismatch."
        )

    if (
        result.get(
            "ab_intersection"
        )
        != 483
    ):

        raise QueueError(
            f"Seed {seed}: "
            "A/B intersection != 483"
        )

    preflight_json = (
        PREFLIGHT_DIR
        / f"model_a_seed{seed}_a9_preflight.json"
    )

    atomic_json(
        preflight_json,
        result,
    )

    config_a, _ = (
        resolve_a9_runtime_configs(
            CONFIG_BASE,
            architecture=
                ARCHITECTURE,
            run_seed=
                seed,
            mutants_hdf5=
                MUTANTS_HDF5,
            wt_hdf5=
                WT_HDF5,
            output_root=
                A_OUTPUT_ROOT,
            peer_output_root=
                B_OUTPUT_ROOT,
            repo_root=
                REPO_DIR,
        )
    )

    if (
        config_a["project"]["seed"]
        != seed
    ):

        raise QueueError(
            f"Seed {seed}: "
            "runtime run seed mismatch."
        )

    if (
        config_a["split"]["seed"]
        != SPLIT_SEED
    ):

        raise QueueError(
            f"Seed {seed}: "
            "runtime split seed mismatch."
        )

    if (
        config_a["model"][
            "architecture"
        ]
        != ARCHITECTURE
    ):

        raise QueueError(
            f"Seed {seed}: "
            "runtime architecture mismatch."
        )

    if (
        config_a["training"][
            "epochs"
        ]
        != 100
    ):

        raise QueueError(
            f"Seed {seed}: "
            "epochs != 100"
        )

    if (
        config_a["training"][
            "batch_size"
        ]
        != 4
    ):

        raise QueueError(
            f"Seed {seed}: "
            "batch_size != 4"
        )

    if (
        config_a["training"][
            "device"
        ]
        != "cuda"
    ):

        raise QueueError(
            f"Seed {seed}: "
            "device != cuda"
        )

    runtime_path = (
        runtime_config_path(
            seed
        )
    )

    save_config(
        config_a,
        runtime_path,
    )

    print(
        f"[seed {seed}] "
        "A9_PREFLIGHT=PASS",
        flush=True,
    )

    print(
        f"[seed {seed}] "
        f"RUNTIME_CONFIG="
        f"{runtime_path}",
        flush=True,
    )

    return runtime_path


# =============================================================================
# LAUNCH FRESH
# =============================================================================

def launch_fresh_seed(
    seed: int,
    runtime_path: Path,
):

    if (
        find_seed_runs(seed)
    ):

        raise QueueError(
            f"Seed {seed}: "
            "run detectado antes "
            "del launch."
        )

    trainers = (
        get_active_trainers()
    )

    if trainers:

        raise QueueError(
            f"Seed {seed}: "
            "otro train.py activo."
        )

    before_runs = set(
        ARCH_ROOT.glob(
            "run_*"
        )
    ) if ARCH_ROOT.exists() else set()

    log_path = (
        training_log_path(
            seed
        )
    )

    launch_path = (
        launcher_path(
            seed
        )
    )

    if launch_path.exists():

        raise QueueError(
            f"Seed {seed}: "
            "launcher persistente "
            "ya existe."
        )

    command = [
        sys.executable,
        "-u",
        "scripts/train.py",
        "--config",
        str(runtime_path),
        "--device",
        "cuda",
    ]

    print(
        "=" * 90,
        flush=True,
    )

    print(
        f"LANZANDO SEED {seed}",
        flush=True,
    )

    print(
        " ".join(command),
        flush=True,
    )

    print(
        "=" * 90,
        flush=True,
    )

    log_handle = open(
        log_path,
        "w",
        encoding="utf-8",
        buffering=1,
    )

    process = subprocess.Popen(
        command,
        cwd=str(REPO_DIR),
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    new_run = None

    try:

        for _ in range(120):

            current_runs = set(
                ARCH_ROOT.glob(
                    "run_*"
                )
            ) if ARCH_ROOT.exists() else set()

            created = sorted(
                current_runs
                - before_runs
            )

            if len(created) > 1:

                try:
                    os.kill(
                        process.pid,
                        signal.SIGINT,
                    )
                except ProcessLookupError:
                    pass

                raise QueueError(
                    f"Seed {seed}: "
                    "aparecieron múltiples "
                    "run_* nuevos."
                )

            if len(created) == 1:

                candidate = (
                    created[0]
                )

                manifest_path = (
                    candidate
                    / "run_manifest.json"
                )

                if manifest_path.is_file():

                    try:

                        manifest = (
                            read_json(
                                manifest_path
                            )
                        )

                        manifest_seed = (
                            manifest
                            .get(
                                "configuration",
                                {},
                            )
                            .get("seed")
                        )

                        if (
                            manifest_seed
                            == seed
                        ):

                            new_run = (
                                candidate
                            )

                            break

                    except Exception:

                        pass

            if (
                process.poll()
                is not None
            ):

                break

            time.sleep(1)

    finally:

        log_handle.close()

    if new_run is None:

        if (
            process.poll()
            is None
        ):

            try:

                os.kill(
                    process.pid,
                    signal.SIGINT,
                )

            except ProcessLookupError:

                pass

        raise QueueError(
            f"Seed {seed}: "
            "no se pudo identificar "
            "inequívocamente el nuevo run."
        )

    launch_record = {
        "architecture":
            ARCHITECTURE,
        "run_seed":
            seed,
        "split_seed":
            SPLIT_SEED,
        "pid":
            process.pid,
        "run_dir":
            str(new_run),
        "runtime_config":
            str(runtime_path),
        "log":
            str(log_path),
        "launched_at_utc":
            utcnow(),
        "launched_by":
            "a9_multiseed_queue",
    }

    atomic_json(
        launch_path,
        launch_record,
    )

    print(
        f"[seed {seed}] "
        f"STARTED PID="
        f"{process.pid}",
        flush=True,
    )

    print(
        f"[seed {seed}] "
        f"RUN_DIR="
        f"{new_run}",
        flush=True,
    )

    return (
        new_run,
        process,
    )


# =============================================================================
# MONITOR
# =============================================================================

def monitor_seed(
    seed: int,
    run_dir: Path,
    process=None,
):

    print(
        f"[seed {seed}] "
        f"MONITORING "
        f"{run_dir}",
        flush=True,
    )

    missing_trainer_checks = 0

    last_reported_epoch = None

    while True:

        manifest_path = (
            run_dir
            / "run_manifest.json"
        )

        if not (
            manifest_path.is_file()
        ):

            raise QueueError(
                f"Seed {seed}: "
                "manifest desapareció."
            )

        manifest = read_json(
            manifest_path
        )

        status = manifest.get(
            "status"
        )

        stage = (
            manifest
            .get(
                "lifecycle",
                {},
            )
            .get("stage")
        )

        training = manifest.get(
            "training",
            {},
        )

        epochs_completed = (
            training.get(
                "epochs_completed"
            )
        )

        rows = read_metrics(
            run_dir
        )

        bad = nonfinite_epochs(
            rows
        )

        if bad:

            update_state(
                status=
                    "stopped_on_error",
                current_seed=
                    seed,
                current_run_dir=
                    str(run_dir),
                error=
                    f"NaN/Inf epochs "
                    f"{bad}",
            )

            interrupt_seed_if_running(
                seed
            )

            raise QueueError(
                f"Seed {seed}: "
                f"NaN/Inf detectado "
                f"en {bad}"
            )

        log_errors = (
            serious_log_patterns(
                training_log_path(
                    seed
                )
            )
        )

        if log_errors:

            update_state(
                status=
                    "stopped_on_error",
                current_seed=
                    seed,
                current_run_dir=
                    str(run_dir),
                error=
                    f"serious log "
                    f"{log_errors}",
            )

            interrupt_seed_if_running(
                seed
            )

            raise QueueError(
                f"Seed {seed}: "
                "error serio en log: "
                f"{log_errors}"
            )

        if (
            epochs_completed
            != last_reported_epoch
        ):

            print(
                f"[seed {seed}] "
                f"status={status} "
                f"stage={stage} "
                f"epochs="
                f"{epochs_completed} "
                f"metrics="
                f"{len(rows)}",
                flush=True,
            )

            last_reported_epoch = (
                epochs_completed
            )

        update_state(
            status=
                "running",
            current_seed=
                seed,
            current_run_dir=
                str(run_dir),
            current_manifest_status=
                status,
            current_stage=
                stage,
            current_epochs_completed=
                epochs_completed,
            current_metric_records=
                len(rows),
        )

        if status == "completed":

            # Dejar que train.py termine
            # completamente antes de lanzar
            # la siguiente seed.

            wait_no_trainers(
                timeout_seconds=120
            )

            return completed_gate(
                run_dir,
                seed,
            )

        if status in {
            "failed",
            "interrupted",
        }:

            raise QueueError(
                f"Seed {seed}: "
                f"manifest status="
                f"{status}"
            )

        trainers = (
            get_active_trainers()
        )

        if len(trainers) > 1:

            raise QueueError(
                f"Seed {seed}: "
                "más de un train.py activo."
            )

        if len(trainers) == 1:

            trainer = trainers[0]

            if not trainer_matches_seed(
                trainer,
                seed,
            ):

                raise QueueError(
                    f"Seed {seed}: "
                    "el trainer activo "
                    "pertenece a otra seed: "
                    f"{trainer}"
                )

            missing_trainer_checks = 0

        else:

            if status == "running":

                missing_trainer_checks += 1

                if (
                    missing_trainer_checks
                    >= 3
                ):

                    raise QueueError(
                        f"Seed {seed}: "
                        "manifest=running "
                        "pero no existe "
                        "train.py durante "
                        "3 comprobaciones."
                    )

        if process is not None:

            # poll() además recoge
            # el proceso cuando finaliza.
            process.poll()

        time.sleep(
            MONITOR_INTERVAL_SECONDS
        )


# =============================================================================
# PROCESAR UNA SEED
# =============================================================================

def process_seed(
    seed: int,
):

    runs = find_seed_runs(
        seed
    )

    if len(runs) > 1:

        raise QueueError(
            f"Seed {seed}: "
            f"hay {len(runs)} runs; "
            "ambigüedad bloqueada."
        )

    if len(runs) == 1:

        run_dir = runs[0]

        manifest = read_json(
            run_dir
            / "run_manifest.json"
        )

        status = manifest.get(
            "status"
        )

        print(
            f"[seed {seed}] "
            f"RUN EXISTENTE "
            f"status={status}",
            flush=True,
        )

        if status == "completed":

            wait_no_trainers(
                timeout_seconds=120
            )

            return completed_gate(
                run_dir,
                seed,
            )

        if status == "running":

            trainers = (
                get_active_trainers()
            )

            matching = [
                trainer
                for trainer
                in trainers
                if trainer_matches_seed(
                    trainer,
                    seed,
                )
            ]

            if len(trainers) != 1:

                raise QueueError(
                    f"Seed {seed}: "
                    "run está running, "
                    "pero el número total "
                    f"de trainers es "
                    f"{len(trainers)}."
                )

            if len(matching) != 1:

                raise QueueError(
                    f"Seed {seed}: "
                    "no se puede asociar "
                    "inequívocamente el "
                    "trainer activo."
                )

            # Adoptar el run actual.
            # Es exactamente el caso A23.

            return monitor_seed(
                seed,
                run_dir,
                process=None,
            )

        raise QueueError(
            f"Seed {seed}: "
            f"run existente con "
            f"status={status!r}. "
            "No se relanza automáticamente."
        )

    # ---------------------------------------------------------
    # No hay run.
    #
    # Para A23 esto sería anómalo.
    # Para A37/A41/A53 creamos fresh.
    # ---------------------------------------------------------

    if seed == 23:

        raise QueueError(
            "A23 debería existir ya; "
            "no se permite crear "
            "un A23 fresh desde la cola."
        )

    runtime_path = (
        prepare_fresh_seed(
            seed
        )
    )

    run_dir, process = (
        launch_fresh_seed(
            seed,
            runtime_path,
        )
    )

    return monitor_seed(
        seed,
        run_dir,
        process=process,
    )


# =============================================================================
# MAIN
# =============================================================================

def main():

    print(
        "=" * 100,
        flush=True,
    )

    print(
        "MODELO A · A9 "
        "MULTISEED QUEUE START",
        flush=True,
    )

    print(
        "SECUENCIA:",
        SEEDS,
        flush=True,
    )

    print(
        "=" * 100,
        flush=True,
    )

    a11_run = (
        validate_a11_gate()
    )

    update_state(
        status="running",
        a11_gate="ACCEPTED",
        a11_run_dir=
            str(a11_run),
    )

    completed = []

    summaries = {}

    for seed in SEEDS:

        update_state(
            status="running",
            current_seed=seed,
            current_run_dir=None,
            completed_seeds=
                completed,
        )

        summary = process_seed(
            seed
        )

        completed.append(
            seed
        )

        summaries[str(seed)] = (
            summary
        )

        update_state(
            status="running",
            current_seed=None,
            current_run_dir=None,
            completed_seeds=
                completed,
            summaries=
                summaries,
        )

    update_state(
        status=
            "training_queue_completed",
        current_seed=None,
        current_run_dir=None,
        completed_seeds=
            completed,
        summaries=
            summaries,
        completed_at_utc=
            utcnow(),
    )

    print(
        "=" * 100,
        flush=True,
    )

    print(
        "A9_MODEL_A_TRAINING_QUEUE="
        "COMPLETE",
        flush=True,
    )

    print(
        "COMPLETED_SEEDS=",
        completed,
        flush=True,
    )

    print(
        "NEXT="
        "POST_TRAINING_EXPORT_AND_ACCEPTANCE",
        flush=True,
    )

    print(
        "=" * 100,
        flush=True,
    )


if __name__ == "__main__":

    try:

        main()

    except BaseException as exc:

        update_state(
            status=
                "stopped_on_error",
            error=
                f"{type(exc).__name__}: "
                f"{exc}",
            traceback=
                traceback.format_exc(),
        )

        print(
            "=" * 100,
            flush=True,
        )

        print(
            "A9_MODEL_A_TRAINING_QUEUE="
            "STOPPED_ON_ERROR",
            flush=True,
        )

        print(
            f"{type(exc).__name__}: "
            f"{exc}",
            flush=True,
        )

        traceback.print_exc()

        print(
            "NO SE LANZARÁ "
            "NINGUNA SEED POSTERIOR.",
            flush=True,
        )

        print(
            "=" * 100,
            flush=True,
        )

        raise
'''

SUPERVISOR_SCRIPT.write_text(
    textwrap.dedent(
        supervisor_code
    ),
    encoding="utf-8",
)

print(
    "SUPERVISOR_SCRIPT =",
    SUPERVISOR_SCRIPT,
)

# =============================================================================
# 2. LANZAR SUPERVISOR EN SEGUNDO PLANO
# =============================================================================

log_handle = open(
    QUEUE_LOG,
    "a",
    encoding="utf-8",
    buffering=1,
)

log_handle.write(
    "\n\n"
    + "=" * 100
    + "\n"
    + "NEW QUEUE SUPERVISOR LAUNCH "
    + datetime.now(
        timezone.utc
    ).isoformat()
    + "\n"
    + "=" * 100
    + "\n"
)

process = subprocess.Popen(
    [
        sys.executable,
        "-u",
        str(
            SUPERVISOR_SCRIPT
        ),
    ],
    cwd=str(REPO_DIR),
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

QUEUE_PID = process.pid

log_handle.close()

LOCAL_PID.write_text(
    str(QUEUE_PID) + "\n",
    encoding="utf-8",
)

launcher_payload = {
    "pid": QUEUE_PID,
    "script":
        str(SUPERVISOR_SCRIPT),
    "log":
        str(QUEUE_LOG),
    "state":
        str(QUEUE_STATE),
    "queue_seeds":
        [23, 37, 41, 53],
    "launched_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

tmp = QUEUE_LAUNCHER.with_name(
    QUEUE_LAUNCHER.name
    + ".tmp"
)

tmp.write_text(
    json.dumps(
        launcher_payload,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

os.replace(
    tmp,
    QUEUE_LAUNCHER,
)

# =============================================================================
# 3. COMPROBAR QUE EL SUPERVISOR NO MUERE EN STARTUP
# =============================================================================

time.sleep(5)

return_code = (
    process.poll()
)

if return_code is not None:

    print()
    print(
        "QUEUE SUPERVISOR "
        "TERMINÓ DURANTE STARTUP."
    )

    print(
        "RETURN_CODE =",
        return_code,
    )

    print()
    print(
        "ÚLTIMAS LÍNEAS DEL LOG:"
    )

    if QUEUE_LOG.is_file():

        lines = (
            QUEUE_LOG
            .read_text(
                encoding="utf-8",
                errors="replace",
            )
            .splitlines()
        )

        for line in lines[-80:]:
            print(line)

    raise RuntimeError(
        "La cola no ha arrancado. "
        "NO lanzar seeds manualmente."
    )

# =============================================================================
# 4. RESULTADO
# =============================================================================

print()
print("=" * 100)
print(
    "A9 MODEL A · MULTISEED QUEUE STARTED"
)
print("=" * 100)

print(
    "QUEUE_PID      =",
    QUEUE_PID,
)

print(
    "QUEUE          =",
    "A23 -> A37 -> A41 -> A53",
)

print(
    "QUEUE_LOG      =",
    QUEUE_LOG,
)

print(
    "QUEUE_STATE    =",
    QUEUE_STATE,
)

print(
    "QUEUE_LAUNCHER =",
    QUEUE_LAUNCHER,
)

print()
print(
    "A23 será ADOPTADO; "
    "NO se reinicia."
)

print(
    "A37/A41/A53 se lanzarán "
    "solo después de que "
    "la seed anterior termine "
    "correctamente."
)

print()
print(
    "A9_MODEL_A_QUEUE=RUNNING"
)

print("=" * 100)

MODELO A · A9 MULTISEED QUEUE
SUPERVISOR_SCRIPT = /content/model_a_workspace/a9_model_a_multiseed_queue_supervisor.py

A9 MODEL A · MULTISEED QUEUE STARTED
QUEUE_PID      = 7515
QUEUE          = A23 -> A37 -> A41 -> A53
QUEUE_LOG      = /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9/_queue/model_a_a9_multiseed_queue.log
QUEUE_STATE    = /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9/_queue/model_a_a9_multiseed_queue_state.json
QUEUE_LAUNCHER = /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9/_queue/model_a_a9_multiseed_queue_launcher.json

A23 será ADOPTADO; NO se reinicia.
A37/A41/A53 se lanzarán solo después de que la seed anterior termine correctamente.

A9_MODEL_A_QUEUE=RUNNING


In [10]:
from pathlib import Path
import subprocess
import json
import sys

# ============================================================
# MODELO A · A9
# ACCEPTANCE MULTISEED
#
# Usa scripts/a9_acceptance.py
# exactamente como en A11.
#
# NO ENTRENA.
# NO MODIFICA CHECKPOINTS.
# ============================================================

REPO_DIR = Path(
    "/content/model_a_workspace/repo"
)

EXPECTED_COMMIT = (
    "3757bfdde52a9920611f3e7b8027434ce80d54bc"
)

A_ROOT = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_a/runs/model_a_a9/"
    "model_a_nodal_multiscale_pair"
)

RUNS = {
    23: (
        A_ROOT
        / "run_20260903T081000.855970Z-4995095d"
    ),
    37: (
        A_ROOT
        / "run_20260903T084337.510387Z-2936e6a4"
    ),
    41: (
        A_ROOT
        / "run_20260903T092748.137509Z-13b67627"
    ),
    53: (
        A_ROOT
        / "run_20260903T101341.325997Z-67f05265"
    ),
}

ACCEPTANCE_SCRIPT = (
    REPO_DIR
    / "scripts"
    / "a9_acceptance.py"
)


def read_json(path):
    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


print("=" * 100)
print("MODELO A · A9 ACCEPTANCE MULTISEED")
print("=" * 100)

# ------------------------------------------------------------
# 0. Identidad del código científico
# ------------------------------------------------------------

assert REPO_DIR.is_dir(), REPO_DIR
assert ACCEPTANCE_SCRIPT.is_file(), ACCEPTANCE_SCRIPT

head = subprocess.run(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "rev-parse",
        "HEAD",
    ],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

dirty = subprocess.run(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "status",
        "--porcelain",
    ],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

print("REPO_HEAD =", head)
print("REPO_CLEAN =", dirty == "")

assert head == EXPECTED_COMMIT
assert dirty == ""

results = {}

# ------------------------------------------------------------
# 1. Acceptance por seed
# ------------------------------------------------------------

for seed in [23, 37, 41, 53]:

    run_dir = RUNS[seed]

    config = (
        run_dir
        / "config_resolved.yaml"
    )

    npz = (
        run_dir
        / "exports"
        / "a9_pair_representations_best.npz"
    )

    ids = (
        run_dir
        / "exports"
        / "a9_pair_representations_variant_ids.txt"
    )

    acceptance = (
        run_dir
        / "a9_acceptance.json"
    )

    print()
    print("=" * 100)
    print(f"SEED {seed} · ACCEPTANCE")
    print("=" * 100)

    assert run_dir.is_dir(), run_dir
    assert config.is_file(), config
    assert npz.is_file() and npz.stat().st_size > 0, npz
    assert ids.is_file() and ids.stat().st_size > 0, ids

    # Fail closed: no sobrescribir una acceptance previa.
    assert not acceptance.exists(), (
        f"Seed {seed}: ya existe {acceptance}. "
        "No se sobrescribirá automáticamente."
    )

    command = [
        sys.executable,
        str(ACCEPTANCE_SCRIPT),
        "--run-dir",
        str(run_dir),
        "--config",
        str(config),
        "--pair-representations",
        str(npz),
    ]

    print(
        "COMMAND =",
        " ".join(command)
    )

    completed = subprocess.run(
        command,
        cwd=str(REPO_DIR),
        text=True,
        capture_output=True,
    )

    print()
    print("RETURN_CODE =", completed.returncode)

    if completed.stdout.strip():
        print()
        print("STDOUT:")
        print(completed.stdout)

    if completed.stderr.strip():
        print()
        print("STDERR:")
        print(completed.stderr)

    assert completed.returncode == 0, (
        f"Acceptance falló para seed {seed}"
    )

    assert acceptance.is_file(), (
        f"Seed {seed}: no se generó a9_acceptance.json"
    )

    payload = read_json(
        acceptance
    )

    status = payload.get(
        "status"
    )

    run_seed = payload.get(
        "run_seed"
    )

    architecture = payload.get(
        "architecture"
    )

    automatic_checks = payload.get(
        "automatic_rejection_checks"
    )

    print("status =", status)
    print("run_seed =", run_seed)
    print("architecture =", architecture)
    print(
        "automatic_rejection_checks =",
        automatic_checks,
    )

    assert status == "accepted"
    assert run_seed == seed
    assert (
        architecture
        == "model_a_nodal_multiscale_pair"
    )
    assert automatic_checks == "PASS"

    reps = payload.get(
        "representation_metrics",
        {}
    )

    assert set(reps) == {
        "h_pair_delta",
        "z_delta_pair",
        "z_instance_pair",
    }

    expected_shapes = {
        "h_pair_delta": [483, 1920],
        "z_delta_pair": [483, 128],
        "z_instance_pair": [483, 64],
    }

    for name, expected_shape in expected_shapes.items():

        metrics = reps[name]

        print()
        print(name)
        print("  shape =", metrics.get("shape"))
        print("  finite =", metrics.get("finite"))
        print(
            "  exact_unique_rows =",
            metrics.get("exact_unique_rows"),
        )
        print(
            "  exact_total_collapse =",
            metrics.get("exact_total_collapse"),
        )
        print(
            "  pc1_variance_fraction =",
            metrics.get("pc1_variance_fraction"),
        )
        print(
            "  effective_rank =",
            metrics.get("effective_rank"),
        )

        assert (
            metrics.get("shape")
            == expected_shape
        )

        assert (
            metrics.get("finite")
            is True
        )

        assert (
            metrics.get("exact_total_collapse")
            is False
        )

        assert (
            metrics.get("exact_unique_rows")
            == 483
        )

    results[seed] = payload

    print()
    print(
        f"A{seed}_ACCEPTANCE=PASS"
    )

# ------------------------------------------------------------
# 2. Resumen multiseed
# ------------------------------------------------------------

print()
print("=" * 100)
print("MODELO A · ACCEPTANCE MULTISEED · RESUMEN")
print("=" * 100)

accepted_seeds = []

for seed in [23, 37, 41, 53]:

    payload = results[seed]

    print(
        f"A{seed}:",
        payload["status"],
    )

    if payload["status"] == "accepted":
        accepted_seeds.append(seed)

print()
print(
    "ACCEPTED_NEW_SEEDS =",
    accepted_seeds
)

assert accepted_seeds == [
    23,
    37,
    41,
    53,
]

print()
print("=" * 100)
print("MODEL_A_MULTISEED_ACCEPTANCE=PASS")
print("=" * 100)

MODELO A · A9 ACCEPTANCE MULTISEED
REPO_HEAD = 3757bfdde52a9920611f3e7b8027434ce80d54bc
REPO_CLEAN = True

SEED 23 · ACCEPTANCE
COMMAND = /usr/bin/python3 /content/model_a_workspace/repo/scripts/a9_acceptance.py --run-dir /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9/model_a_nodal_multiscale_pair/run_20260903T081000.855970Z-4995095d --config /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9/model_a_nodal_multiscale_pair/run_20260903T081000.855970Z-4995095d/config_resolved.yaml --pair-representations /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9/model_a_nodal_multiscale_pair/run_20260903T081000.855970Z-4995095d/exports/a9_pair_representations_best.npz

RETURN_CODE = 0

STDOUT:
{
  "architecture": "model_a_nodal_multiscale_pair",
  "automatic_rejection_checks": "PASS",
  "representation_metrics": {
    "h_pair_delta": {
      "effective_rank": 2.1480013872666333,
      "exact_total_collapse": false,
      "exact_unique_rows

In [9]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import hashlib
import json
import os
import subprocess

import numpy as np
import yaml


# =====================================================================
# MODELO A · A9
# GLOBAL MULTISEED ACCEPTANCE / CLOSURE
# =====================================================================
#
# REUTILIZA:
#   A11/A23/A37/A41/A53 a9_acceptance.json existentes
#
# NO:
#   - entrena
#   - carga train.py
#   - ejecuta a9_acceptance.py
#   - regenera exports
#   - modifica checkpoints
#   - modifica manifests/configs/splits
#
# ÚNICA escritura permitida:
#   model_a_a9_acceptance.json
#
# =====================================================================


EXPECTED_ARCHITECTURE = "model_a_nodal_multiscale_pair"

EXPECTED_SEEDS = [
    11,
    23,
    37,
    41,
    53,
]

EXPECTED_SPLIT_SEED = 42
EXPECTED_VARIANTS = 483
EXPECTED_NATIVE_WT = 1

PROTOCOL_REFERENCE_COMMIT = (
    "3757bfdde52a9920611f3e7b8027434ce80d54bc"
)

EXPECTED_PARTITIONS = {
    "train": 342,
    "validation": 78,
    "test": 63,
}

EXPECTED_REPRESENTATION_SHAPES = {
    "h_pair_delta": [483, 1920],
    "z_delta_pair": [483, 128],
    "z_instance_pair": [483, 64],
}

EXPECTED_HDF5 = {
    "mutants":
        "92eb242f5565db6194a5e29e3469775fbf7d8c07280ed2d8cdfd5ab98b5b5631",

    "wt_companion":
        "29f68e98ae300207511594e0baf7621ba9e67c85d8df12ceee812a1dea3aa91a",

    "combined":
        "9a320862a54566d232e1ef2a4eafa468cf900a19186dc0fd7c92aaf0abe06c68",
}

EXPECTED_SPLIT_FINGERPRINT = (
    "8dcc611a6575b95242fda70d2b29e6bf4917a9c781bbeebd851e95e71391d7dd"
)


# =====================================================================
# PATHS
# =====================================================================

A9_ROOT = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_a/runs/model_a_a9"
)

ARCH_ROOT = (
    A9_ROOT
    / EXPECTED_ARCHITECTURE
)

RUNS = {

    11: (
        ARCH_ROOT
        / "run_20260902T093129.973812Z-5c337698"
    ),

    23: (
        ARCH_ROOT
        / "run_20260903T081000.855970Z-4995095d"
    ),

    37: (
        ARCH_ROOT
        / "run_20260903T084337.510387Z-2936e6a4"
    ),

    41: (
        ARCH_ROOT
        / "run_20260903T092748.137509Z-13b67627"
    ),

    53: (
        ARCH_ROOT
        / "run_20260903T101341.325997Z-67f05265"
    ),
}

GLOBAL_ACCEPTANCE = (
    A9_ROOT
    / "model_a_a9_acceptance.json"
)

TMP_GLOBAL_ACCEPTANCE = (
    A9_ROOT
    / "model_a_a9_acceptance.json.tmp"
)

REPO_DIR = Path(
    "/content/model_a_workspace/repo"
)


# =====================================================================
# HELPERS
# =====================================================================

def read_json(path):

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def read_yaml(path):

    return yaml.safe_load(
        path.read_text(
            encoding="utf-8"
        )
    )


def sha256(path):

    h = hashlib.sha256()

    with path.open("rb") as stream:

        for chunk in iter(
            lambda: stream.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


_MISSING = object()


def get_nested(
    mapping,
    dotted_path,
    default=_MISSING,
):

    value = mapping

    try:

        for part in dotted_path.split("."):
            value = value[part]

    except (KeyError, TypeError):

        if default is not _MISSING:
            return default

        raise AssertionError(
            f"Falta campo requerido: {dotted_path}"
        )

    return value


def hdf5_manifest_fingerprints(
    manifest,
):

    files = get_nested(
        manifest,
        "data.hdf5_content_fingerprint.files",
    )

    assert isinstance(files, list)

    by_role = {}

    for row in files:

        if isinstance(row, dict):

            role = row.get("role")
            digest = row.get("digest")

            if role is not None:
                by_role[str(role)] = digest

    combined = get_nested(
        manifest,
        "data.hdf5_content_fingerprint.combined.digest",
    )

    return {
        "mutants": by_role.get("mutants"),
        "wt_companion": by_role.get("wt_companion"),
        "combined": combined,
    }


def comparable_config_signature(
    config,
):

    """
    Solo campos científicos/productivos que deben ser
    idénticos entre las cinco seeds.

    Se excluyen deliberadamente:
      project.seed
      reproducibility.seed_*
    porque deben cambiar con cada seed.
    """

    paths = [

        # modelo
        "model.architecture",
        "model.active_scales",
        "model.projection_instance.enabled",

        # features
        "features.node_groups",
        "features.edge_groups",
        "features.graph_groups",
        "features.domains.enabled",

        # entrenamiento
        "training.epochs",
        "training.batch_size",
        "training.optimizer",
        "training.learning_rate",
        "training.weight_decay",
        "training.scheduler",
        "training.early_stopping.enabled",
        "training.early_stopping.monitor",
        "training.early_stopping.mode",
        "training.early_stopping.patience",
        "training.early_stopping.min_delta",
        "training.checkpointing.monitor",
        "training.checkpointing.mode",
        "training.checkpointing.save_best",
        "training.checkpointing.save_last",
        "training.mixed_precision.enabled",

        # loss
        "loss.main",
        "loss.temperature",
        "loss.lambda_wt",
        "loss.lambda_delta",
        "loss.false_negative_mask.enabled",
        "loss.false_negative_mask.mode",
        "loss.false_negative_mask.same_position",
        "loss.false_negative_mask.strict",
        "loss.false_negative_mask.min_valid_negatives",
        "loss.false_negative_mask.min_valid_negative_fraction",

        # split
        "split.type",
        "split.seed",
        "split.persist_path",
        "split.allow_create",

        # A9
        "a9.run_seeds",
        "a9.expected_hdf5_fingerprints",
        "a9.expected_split_fingerprint",
    ]

    return {
        path:
            get_nested(
                config,
                path,
            )
        for path in paths
    }


# =====================================================================
# 0. PREFLIGHT GLOBAL
# =====================================================================

print("=" * 110)
print("MODELO A · A9 GLOBAL ACCEPTANCE CORREGIDA")
print("=" * 110)

assert A9_ROOT.is_dir(), A9_ROOT
assert ARCH_ROOT.is_dir(), ARCH_ROOT

# No sobrescribir un cierre global previo.
assert not GLOBAL_ACCEPTANCE.exists(), (
    f"Ya existe {GLOBAL_ACCEPTANCE}. "
    "No se sobrescribirá."
)

assert not TMP_GLOBAL_ACCEPTANCE.exists(), (
    f"Existe temporal inesperado: "
    f"{TMP_GLOBAL_ACCEPTANCE}"
)

print("A9_ROOT =", A9_ROOT)
print("ARCH_ROOT =", ARCH_ROOT)
print("GLOBAL_ACCEPTANCE =", GLOBAL_ACCEPTANCE)

print()
print("GLOBAL_PREFLIGHT=PASS")


# =====================================================================
# 1. PROCEDENCIA DEL CÓDIGO DEL CIERRE
#
# git_state.json NO es requisito de los runs.
#
# Si el workspace sigue disponible, verificamos HEAD y clean.
# Si Colab fue reiniciado y /content desapareció, NO bloqueamos:
# queda registrado de forma transparente.
# =====================================================================

print()
print("=" * 110)
print("PROCEDENCIA DEL CÓDIGO")
print("=" * 110)

repo_provenance = {
    "protocol_reference_commit":
        PROTOCOL_REFERENCE_COMMIT,

    "current_runtime_repository_available":
        False,

    "current_runtime_repository_verified":
        False,

    "note":
        (
            "run-level git_state.json is not part "
            "of the official A9 acceptance contract"
        ),
}


if (
    REPO_DIR.is_dir()
    and
    (REPO_DIR / ".git").exists()
):

    head = subprocess.run(
        [
            "git",
            "-C",
            str(REPO_DIR),
            "rev-parse",
            "HEAD",
        ],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()

    dirty = subprocess.run(
        [
            "git",
            "-C",
            str(REPO_DIR),
            "status",
            "--porcelain",
        ],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()

    print("CURRENT_REPO_HEAD =", head)
    print("CURRENT_REPO_CLEAN =", dirty == "")

    assert (
        head
        == PROTOCOL_REFERENCE_COMMIT
    )

    assert dirty == ""

    repo_provenance[
        "current_runtime_repository_available"
    ] = True

    repo_provenance[
        "current_runtime_repository_verified"
    ] = True

    repo_provenance[
        "current_runtime_head"
    ] = head

    repo_provenance[
        "current_runtime_clean"
    ] = True

    print(
        "CURRENT_RUNTIME_CODE_PROVENANCE=PASS"
    )

else:

    print(
        "CURRENT_RUNTIME_REPO=NOT_AVAILABLE"
    )

    print(
        "INFO: Colab /content puede haberse reiniciado."
    )

    print(
        "INFO: esto NO invalida las acceptances "
        "ya persistidas en Drive."
    )

    repo_provenance[
        "current_runtime_note"
    ] = (
        "Ephemeral Colab repository unavailable "
        "during global materialization."
    )


# =====================================================================
# 2. UNICIDAD DE RUN PRODUCTIVO COMPLETED
# =====================================================================

print()
print("=" * 110)
print("UNICIDAD DE RUNS PRODUCTIVOS COMPLETED")
print("=" * 110)

completed_by_seed = {
    seed: []
    for seed in EXPECTED_SEEDS
}


for candidate in sorted(
    ARCH_ROOT.glob("run_*")
):

    manifest_path = (
        candidate
        / "run_manifest.json"
    )

    if not manifest_path.is_file():
        continue

    try:
        manifest = read_json(
            manifest_path
        )
    except Exception:
        continue

    architecture = manifest.get(
        "architecture"
    )

    status = manifest.get(
        "status"
    )

    seed = get_nested(
        manifest,
        "configuration.seed",
        default=None,
    )

    if (
        architecture
        == EXPECTED_ARCHITECTURE
        and
        status == "completed"
        and
        seed in EXPECTED_SEEDS
    ):

        completed_by_seed[
            seed
        ].append(
            candidate.resolve()
        )


for seed in EXPECTED_SEEDS:

    candidates = (
        completed_by_seed[seed]
    )

    print(
        f"SEED {seed}: "
        f"{len(candidates)} completed run(s)"
    )

    for candidate in candidates:
        print(" ", candidate)

    assert len(candidates) == 1, (
        f"Seed {seed}: se esperaba exactamente "
        "un run completed."
    )

    assert (
        candidates[0]
        == RUNS[seed].resolve()
    ), (
        f"Seed {seed}: el único run completed "
        "no coincide con el run canónico."
    )


print()
print(
    "PRODUCTIVE_RUN_UNIQUENESS=PASS"
)


# =====================================================================
# 3. A11 = REFERENCIA DEL ORDEN DE LAS 483 VARIANTES
# =====================================================================

A11_IDS_FILE = (
    RUNS[11]
    / "exports"
    / "a9_pair_representations_variant_ids.txt"
)

assert A11_IDS_FILE.is_file()

REFERENCE_IDS = np.asarray(
    [
        line.strip()

        for line in A11_IDS_FILE.read_text(
            encoding="utf-8"
        ).splitlines()

        if line.strip()
    ],
    dtype=str,
)

assert (
    REFERENCE_IDS.shape
    == (EXPECTED_VARIANTS,)
)

assert (
    len(
        set(
            REFERENCE_IDS.tolist()
        )
    )
    == EXPECTED_VARIANTS
)

print()
print(
    "A11_REFERENCE_VARIANT_IDS=PASS"
)


# =====================================================================
# 4. VALIDACIÓN GLOBAL DE A11/A23/A37/A41/A53
# =====================================================================

run_records = {}

reference_signature = None

split_file_hashes = set()
split_fingerprints = set()

hdf5_signatures = set()


for seed in EXPECTED_SEEDS:

    print()
    print("=" * 110)
    print(
        f"SEED {seed} · GLOBAL COMPONENT CHECK"
    )
    print("=" * 110)

    run_dir = RUNS[seed]

    manifest_path = (
        run_dir
        / "run_manifest.json"
    )

    config_path = (
        run_dir
        / "config_resolved.yaml"
    )

    split_path = (
        run_dir
        / "split.json"
    )

    gradient_path = (
        run_dir
        / "gradient_audit.json"
    )

    best_path = (
        run_dir
        / "checkpoints"
        / "best.pt"
    )

    last_path = (
        run_dir
        / "checkpoints"
        / "last.pt"
    )

    acceptance_path = (
        run_dir
        / "a9_acceptance.json"
    )

    npz_path = (
        run_dir
        / "exports"
        / "a9_pair_representations_best.npz"
    )

    ids_path = (
        run_dir
        / "exports"
        / "a9_pair_representations_variant_ids.txt"
    )


    # ---------------------------------------------------------
    # 4A. Artefactos obligatorios
    # ---------------------------------------------------------

    required_files = [
        manifest_path,
        config_path,
        split_path,
        gradient_path,
        best_path,
        last_path,
        acceptance_path,
        npz_path,
        ids_path,
    ]

    for path in required_files:

        assert (
            path.is_file()
            and
            path.stat().st_size > 0
        ), (
            f"Seed {seed}: artefacto ausente/vacío: "
            f"{path}"
        )


    # ---------------------------------------------------------
    # 4B. Manifest
    # ---------------------------------------------------------

    manifest = read_json(
        manifest_path
    )

    assert (
        manifest.get("status")
        == "completed"
    )

    assert (
        manifest.get("architecture")
        == EXPECTED_ARCHITECTURE
    )

    assert (
        get_nested(
            manifest,
            "configuration.seed",
        )
        == seed
    )

    assert (
        get_nested(
            manifest,
            "configuration.seed_bundle.split",
        )
        == EXPECTED_SPLIT_SEED
    )

    assert (
        get_nested(
            manifest,
            "data.inventory.biological_variants",
        )
        == EXPECTED_VARIANTS
    )

    assert (
        get_nested(
            manifest,
            "data.inventory.native_wt_controls",
        )
        == EXPECTED_NATIVE_WT
    )


    # ---------------------------------------------------------
    # 4C. HDF5 fingerprints
    # ---------------------------------------------------------

    manifest_hdf5 = (
        hdf5_manifest_fingerprints(
            manifest
        )
    )

    assert (
        manifest_hdf5
        == EXPECTED_HDF5
    ), (
        seed,
        manifest_hdf5,
    )

    hdf5_signatures.add(
        json.dumps(
            manifest_hdf5,
            sort_keys=True,
        )
    )


    # ---------------------------------------------------------
    # 4D. Split fingerprint de manifest
    # ---------------------------------------------------------

    manifest_split_fp = get_nested(
        manifest,
        "data.split_fingerprint",
    )

    assert (
        manifest_split_fp
        == EXPECTED_SPLIT_FINGERPRINT
    )

    split_fingerprints.add(
        manifest_split_fp
    )


    # ---------------------------------------------------------
    # 4E. Acceptance EXISTENTE
    #
    # No se recalcula.
    # ---------------------------------------------------------

    acceptance = read_json(
        acceptance_path
    )

    assert (
        acceptance.get("status")
        == "accepted"
    )

    assert (
        acceptance.get("run_seed")
        == seed
    )

    assert (
        acceptance.get("architecture")
        == EXPECTED_ARCHITECTURE
    )

    assert (
        acceptance.get(
            "automatic_rejection_checks"
        )
        == "PASS"
    )

    reps = acceptance.get(
        "representation_metrics",
        {}
    )

    assert (
        set(reps.keys())
        == set(
            EXPECTED_REPRESENTATION_SHAPES.keys()
        )
    )

    for name, expected_shape in (
        EXPECTED_REPRESENTATION_SHAPES.items()
    ):

        metrics = reps[name]

        assert (
            metrics.get("shape")
            == expected_shape
        )

        assert (
            metrics.get("finite")
            is True
        )

        assert (
            metrics.get(
                "exact_total_collapse"
            )
            is False
        )

        assert (
            metrics.get(
                "exact_unique_rows"
            )
            == EXPECTED_VARIANTS
        )


    # ---------------------------------------------------------
    # 4F. Export existente
    # ---------------------------------------------------------

    with np.load(
        npz_path,
        allow_pickle=False,
    ) as exported:

        assert (
            set(exported.files)
            == set(
                EXPECTED_REPRESENTATION_SHAPES.keys()
            )
        )

        for name, expected_shape in (
            EXPECTED_REPRESENTATION_SHAPES.items()
        ):

            matrix = exported[name]

            assert (
                list(matrix.shape)
                == expected_shape
            )

            assert np.isfinite(
                matrix
            ).all()

            assert (
                np.unique(
                    matrix,
                    axis=0,
                ).shape[0]
                == EXPECTED_VARIANTS
            )


    # ---------------------------------------------------------
    # 4G. Variant IDs
    # ---------------------------------------------------------

    ids = np.asarray(
        [
            line.strip()

            for line in ids_path.read_text(
                encoding="utf-8"
            ).splitlines()

            if line.strip()
        ],
        dtype=str,
    )

    assert (
        ids.shape
        == (EXPECTED_VARIANTS,)
    )

    assert (
        len(
            set(
                ids.tolist()
            )
        )
        == EXPECTED_VARIANTS
    )

    assert np.array_equal(
        ids,
        REFERENCE_IDS,
    ), (
        f"Seed {seed}: el orden de variant_ids "
        "no coincide con A11."
    )


    # ---------------------------------------------------------
    # 4H. Split real
    # ---------------------------------------------------------

    split_payload = read_json(
        split_path
    )

    assignments = (
        split_payload.get(
            "assignments"
        )
    )

    assert isinstance(
        assignments,
        list,
    )

    assert (
        len(assignments)
        == EXPECTED_VARIANTS
    )

    partition_counts = Counter(
        row["partition"]
        for row in assignments
    )

    assert (
        dict(partition_counts)
        == EXPECTED_PARTITIONS
    ), (
        seed,
        dict(partition_counts),
    )


    # Variant IDs no repetidos en split.
    split_variant_ids = [
        str(row["variant_id"])
        for row in assignments
    ]

    assert (
        len(set(split_variant_ids))
        == EXPECTED_VARIANTS
    )


    # No leakage por variante.
    variants_by_partition = {
        partition: {
            str(row["variant_id"])
            for row in assignments
            if row["partition"] == partition
        }
        for partition in EXPECTED_PARTITIONS
    }


    # No leakage por posición.
    positions_by_partition = {
        partition: {
            int(row["position"])
            for row in assignments
            if row["partition"] == partition
        }
        for partition in EXPECTED_PARTITIONS
    }


    pairs = [
        ("train", "validation"),
        ("train", "test"),
        ("validation", "test"),
    ]

    for left, right in pairs:

        assert not (
            variants_by_partition[left]
            &
            variants_by_partition[right]
        ), (
            f"Seed {seed}: variant leakage "
            f"{left}/{right}"
        )

        assert not (
            positions_by_partition[left]
            &
            positions_by_partition[right]
        ), (
            f"Seed {seed}: position leakage "
            f"{left}/{right}"
        )


    # WT nativo no debe estar en split.
    native_wt_ids = {
        str(x)

        for x in get_nested(
            manifest,
            "data.inventory.native_wt_control_ids",
        )
    }

    assert len(native_wt_ids) == 1

    assert not (
        native_wt_ids
        &
        set(split_variant_ids)
    )


    split_hash = sha256(
        split_path
    )

    split_file_hashes.add(
        split_hash
    )


    # ---------------------------------------------------------
    # 4I. Config resuelto
    # ---------------------------------------------------------

    config = read_yaml(
        config_path
    )

    assert (
        get_nested(
            config,
            "project.seed",
        )
        == seed
    )

    for seed_field in [
        "seed_python",
        "seed_numpy",
        "seed_torch",
        "seed_cuda",
        "seed_dataloader",
    ]:

        assert (
            get_nested(
                config,
                f"reproducibility.{seed_field}",
            )
            == seed
        )


    assert (
        get_nested(
            config,
            "model.architecture",
        )
        == EXPECTED_ARCHITECTURE
    )

    assert (
        get_nested(
            config,
            "model.active_scales",
        )
        == [
            "mutation",
            "local",
            "global",
        ]
    )

    assert (
        get_nested(
            config,
            "model.projection_instance.enabled",
        )
        is False
    )

    assert (
        get_nested(
            config,
            "features.domains.enabled",
        )
        is False
    )


    # ---------------------------------------------------------
    # Contrato productivo A9
    # ---------------------------------------------------------

    required_config_values = {

        "training.epochs": 100,
        "training.batch_size": 4,
        "training.optimizer": "adamw",
        "training.learning_rate": 0.001,
        "training.weight_decay": 0.0001,
        "training.scheduler": "cosine",

        "training.early_stopping.enabled": True,
        "training.early_stopping.monitor":
            "validation_loss",
        "training.early_stopping.mode": "min",
        "training.early_stopping.patience": 15,
        "training.early_stopping.min_delta": 0.0,

        "training.checkpointing.monitor":
            "validation_loss",
        "training.checkpointing.mode": "min",
        "training.checkpointing.save_best": True,
        "training.checkpointing.save_last": True,

        "training.mixed_precision.enabled": False,

        "loss.main": "nt_xent",
        "loss.temperature": 0.2,

        "loss.false_negative_mask.enabled": True,
        "loss.false_negative_mask.mode":
            "same_position",
        "loss.false_negative_mask.same_position": True,
        "loss.false_negative_mask.strict": True,
        "loss.false_negative_mask.min_valid_negatives": 1,
        "loss.false_negative_mask.min_valid_negative_fraction":
            0.0,

        "loss.lambda_wt": 0.0,
        "loss.lambda_delta": 0.0,

        "split.type": "leave_position_out",
        "split.seed": EXPECTED_SPLIT_SEED,
        "split.persist_path":
            "splits/leave_position_out_seed_42.json",
        "split.allow_create": False,
    }


    for path, expected in (
        required_config_values.items()
    ):

        actual = get_nested(
            config,
            path,
        )

        assert actual == expected, (
            f"Seed {seed}: {path}: "
            f"esperado={expected!r}, "
            f"actual={actual!r}"
        )


    assert (
        get_nested(
            config,
            "a9.run_seeds",
        )
        == EXPECTED_SEEDS
    )

    assert (
        get_nested(
            config,
            "a9.expected_hdf5_fingerprints",
        )
        == EXPECTED_HDF5
    )

    assert (
        get_nested(
            config,
            "a9.expected_split_fingerprint",
        )
        == EXPECTED_SPLIT_FINGERPRINT
    )


    # ---------------------------------------------------------
    # 4J. Configuración comparable entre seeds
    # ---------------------------------------------------------

    signature = (
        comparable_config_signature(
            config
        )
    )

    if reference_signature is None:

        reference_signature = (
            signature
        )

    else:

        assert (
            signature
            == reference_signature
        ), (
            f"Seed {seed}: configuración "
            "científica distinta del resto."
        )


    # ---------------------------------------------------------
    # 4K. Registro del componente
    # ---------------------------------------------------------

    run_records[
        str(seed)
    ] = {

        "seed":
            seed,

        "run_dir":
            str(run_dir),

        "run_manifest":
            str(manifest_path),

        "run_manifest_sha256":
            sha256(manifest_path),

        "config_resolved":
            str(config_path),

        "config_resolved_sha256":
            sha256(config_path),

        "split":
            str(split_path),

        "split_sha256":
            split_hash,

        "gradient_audit":
            str(gradient_path),

        "gradient_audit_sha256":
            sha256(gradient_path),

        "best_checkpoint":
            str(best_path),

        "best_checkpoint_sha256":
            sha256(best_path),

        "last_checkpoint":
            str(last_path),

        "last_checkpoint_sha256":
            sha256(last_path),

        "pair_representations":
            str(npz_path),

        "pair_representations_sha256":
            sha256(npz_path),

        "variant_ids":
            str(ids_path),

        "variant_ids_sha256":
            sha256(ids_path),

        "individual_acceptance":
            str(acceptance_path),

        "individual_acceptance_sha256":
            sha256(acceptance_path),

        "individual_acceptance_status":
            "accepted",

        "global_component_status":
            "PASS",
    }


    print(
        f"A{seed}_GLOBAL_COMPONENT=PASS"
    )


# =====================================================================
# 5. CONSISTENCIA ENTRE LAS CINCO SEEDS
# =====================================================================

print()
print("=" * 110)
print("CONSISTENCIA MULTISEED")
print("=" * 110)


assert (
    len(split_file_hashes)
    == 1
), (
    "Los split.json no son byte-idénticos "
    "entre seeds."
)

assert (
    split_fingerprints
    == {
        EXPECTED_SPLIT_FINGERPRINT
    }
)

assert (
    len(hdf5_signatures)
    == 1
)

accepted_seeds = [

    seed

    for seed in EXPECTED_SEEDS

    if run_records[
        str(seed)
    ][
        "individual_acceptance_status"
    ]
    == "accepted"
]


assert (
    accepted_seeds
    == EXPECTED_SEEDS
)


print(
    "EXPECTED_SEEDS =",
    EXPECTED_SEEDS
)

print(
    "ACCEPTED_SEEDS =",
    accepted_seeds
)

print(
    "COMMON_SPLIT_SHA256 =",
    next(
        iter(
            split_file_hashes
        )
    )
)

print(
    "SPLIT_FINGERPRINT =",
    EXPECTED_SPLIT_FINGERPRINT
)

print(
    "HDF5_FINGERPRINTS =",
    EXPECTED_HDF5
)

print(
    "VARIANT_ID_ORDER_ACROSS_SEEDS=PASS"
)

print(
    "SCIENTIFIC_CONFIG_ACROSS_SEEDS=PASS"
)

print(
    "DATASET_IDENTITY_ACROSS_SEEDS=PASS"
)

print(
    "SPLIT_IDENTITY_ACROSS_SEEDS=PASS"
)

print(
    "ALL_INDIVIDUAL_ACCEPTANCES=PASS"
)


# =====================================================================
# 6. MATERIALIZAR ÚNICAMENTE EL REGISTRO GLOBAL
# =====================================================================

global_payload = {

    "status":
        "accepted",

    "closure_status":
        "CLOSED",

    "protocol":
        "A9_multiseed_global_closure",

    "architecture":
        EXPECTED_ARCHITECTURE,

    "expected_seeds":
        EXPECTED_SEEDS,

    "accepted_seeds":
        accepted_seeds,

    "individual_acceptances_reused":
        True,

    "individual_acceptances_reexecuted":
        False,

    "training_reexecuted":
        False,

    "exports_reexecuted":
        False,

    "protocol_reference_commit":
        PROTOCOL_REFERENCE_COMMIT,

    "code_provenance":
        repo_provenance,

    "dataset_fingerprints":
        EXPECTED_HDF5,

    "split_seed":
        EXPECTED_SPLIT_SEED,

    "split_fingerprint":
        EXPECTED_SPLIT_FINGERPRINT,

    "split_json_sha256":
        next(
            iter(
                split_file_hashes
            )
        ),

    "partition_counts":
        EXPECTED_PARTITIONS,

    "biological_variants":
        EXPECTED_VARIANTS,

    "native_wt_controls":
        EXPECTED_NATIVE_WT,

    "variant_id_order_consistent":
        True,

    "scientific_configuration_consistent":
        True,

    "runs":
        run_records,

    "materialized_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


# =====================================================================
# 7. ESCRITURA ATÓMICA
# =====================================================================

TMP_GLOBAL_ACCEPTANCE.write_text(

    json.dumps(
        global_payload,
        indent=2,
        sort_keys=True,
    )
    + "\n",

    encoding="utf-8",
)


# Releer y validar ANTES de publicar.
check = read_json(
    TMP_GLOBAL_ACCEPTANCE
)

assert (
    check["status"]
    == "accepted"
)

assert (
    check["closure_status"]
    == "CLOSED"
)

assert (
    check["accepted_seeds"]
    == EXPECTED_SEEDS
)

assert (
    check[
        "individual_acceptances_reused"
    ]
    is True
)

assert (
    check[
        "individual_acceptances_reexecuted"
    ]
    is False
)

assert (
    check[
        "training_reexecuted"
    ]
    is False
)

assert (
    check[
        "exports_reexecuted"
    ]
    is False
)


# Publicación atómica.
os.replace(
    TMP_GLOBAL_ACCEPTANCE,
    GLOBAL_ACCEPTANCE,
)


# =====================================================================
# 8. VERIFICACIÓN POST-ESCRITURA
# =====================================================================

assert (
    GLOBAL_ACCEPTANCE.is_file()
    and
    GLOBAL_ACCEPTANCE.stat().st_size > 0
)

final_payload = read_json(
    GLOBAL_ACCEPTANCE
)

assert (
    final_payload["status"]
    == "accepted"
)

assert (
    final_payload["closure_status"]
    == "CLOSED"
)

assert (
    final_payload["accepted_seeds"]
    == EXPECTED_SEEDS
)


print()
print("=" * 110)
print("MODELO A · A9 GLOBAL ACCEPTANCE · FINAL")
print("=" * 110)

print(
    "GLOBAL_ACCEPTANCE_FILE =",
    GLOBAL_ACCEPTANCE
)

print(
    "GLOBAL_ACCEPTANCE_SHA256 =",
    sha256(
        GLOBAL_ACCEPTANCE
    )
)

print(
    "STATUS =",
    final_payload["status"]
)

print(
    "CLOSURE_STATUS =",
    final_payload[
        "closure_status"
    ]
)

print(
    "ACCEPTED_SEEDS =",
    final_payload[
        "accepted_seeds"
    ]
)

print()

for seed in EXPECTED_SEEDS:

    print(
        f"A{seed}_ACCEPTANCE=PASS"
    )

print()
print(
    "MODEL_A_A9_GLOBAL_ACCEPTANCE=PASS"
)

print(
    "MODEL_A_A9=CLOSED"
)

print("=" * 110)

MODELO A · A9 GLOBAL ACCEPTANCE CORREGIDA
A9_ROOT = /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9
ARCH_ROOT = /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9/model_a_nodal_multiscale_pair
GLOBAL_ACCEPTANCE = /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9/model_a_a9_acceptance.json

GLOBAL_PREFLIGHT=PASS

PROCEDENCIA DEL CÓDIGO
CURRENT_REPO_HEAD = 3757bfdde52a9920611f3e7b8027434ce80d54bc
CURRENT_REPO_CLEAN = True
CURRENT_RUNTIME_CODE_PROVENANCE=PASS

UNICIDAD DE RUNS PRODUCTIVOS COMPLETED
SEED 11: 1 completed run(s)
  /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9/model_a_nodal_multiscale_pair/run_20260902T093129.973812Z-5c337698
SEED 23: 1 completed run(s)
  /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9/model_a_nodal_multiscale_pair/run_20260903T081000.855970Z-4995095d
SEED 37: 1 completed run(s)
  /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9/model_a_no